In [1]:
import pandas as pd
import numpy as np
import os
import torch
import librosa
import noisereduce as nr
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from transformers import AutoTokenizer, AutoModel
import warnings
import pickle
from collections import defaultdict

warnings.filterwarnings('ignore')

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
EXCEPTION_NUMBER = [
    '451', '458', '480'           # Ellie 발화 누락 (3명)
]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 하이퍼파라미터
MAX_UTTERANCE_DURATION = 15.0   # 최대 발화 길이 (초) - 이상 분할
MIN_UTTERANCE_DURATION = 0.5    # 최소 발화 길이 (초) - 이하 제거
SR = 16000

# ⭐ 텍스트 특징 추출 옵션
EXTRACT_TEXT_FEATURES = True
TEXT_MODEL_NAME = "roberta-base"  # MentalBERT
# 대안: "bert-base-uncased", "roberta-base", "distilbert-base-uncased"

# Question Type 매핑 (단순화)
Q_TYPE_MAPPING = {
    'casual': 0,      # small talk, preference, open-ended encouragement
    'background': 1,  # daily habits, social, self-perception
    'emotional': 2,   # emotion / mood
    'clinical': 3,    # depression symptoms direct
    'other': 4
}

# 원본 → 단순화 매핑
Q_TYPE_SIMPLIFICATION = {
    'small talk': 'casual',
    'preference': 'casual',
    'open-ended encouragement': 'casual',
    
    'daily habits / lifestyle': 'background',
    'social / family / relationship': 'background',
    'self-perception / personality': 'background',
    
    'emotion / mood': 'emotional',
    
    'depression symptoms direct': 'clinical',
    
    'other': 'other'
}

print(f"⏳ Wav2Vec 2.0 모델 로딩 중... (Device: {DEVICE})")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
wav2vec_model.eval()
print("✅ Wav2Vec 모델 로드 완료!")

# ⭐ 텍스트 모델 로딩
if EXTRACT_TEXT_FEATURES:
    print(f"⏳ 텍스트 모델 로딩 중: {TEXT_MODEL_NAME}")
    try:
        text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
        text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
        text_model.eval()
        print("✅ 텍스트 모델 로드 완료!")
    except Exception as e:
        print(f"⚠️  MentalBERT 로드 실패, BERT로 fallback...")
        TEXT_MODEL_NAME = "bert-base-uncased"
        text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
        text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
        text_model.eval()
        print("✅ BERT 모델 로드 완료!")


# =============================================================================
# 텍스트 특징 추출
# =============================================================================
def get_ttr(text):
    """Type-Token Ratio 계산"""
    if not text or len(text.strip()) == 0:
        return 0.0
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0.0
    return len(set(tokens)) / len(tokens)


def extract_text_embedding(text):
    """
    BERT/MentalBERT 임베딩 추출
    
    Returns:
        embedding: [768] (BERT) or [768] (MentalBERT)
    """
    try:
        if not text or len(text.strip()) == 0:
            return None
        
        # Tokenization
        inputs = text_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(DEVICE)
        
        # Forward pass
        with torch.no_grad():
            outputs = text_model(**inputs)
            # [CLS] token 사용 (문장 전체 표현)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze()
            
            # 또는 평균 풀링
            # mean_embedding = torch.mean(outputs.last_hidden_state, dim=1).squeeze()
        
        embedding = cls_embedding.cpu().numpy()
        
        # 안전성 체크
        if not np.isfinite(embedding).all():
            return None
        
        return embedding
    
    except Exception as e:
        print(f"⚠️  텍스트 임베딩 추출 실패: {e}")
        return None


def extract_linguistic_features(text):
    """
    추가 언어학적 특징 추출
    
    Returns:
        dict: {
            'word_count': int,
            'avg_word_length': float,
            'sentence_count': int,
            'negation_count': int,
            'first_person_count': int
        }
    """
    if not text or len(text.strip()) == 0:
        return {
            'word_count': 0,
            'avg_word_length': 0.0,
            'sentence_count': 0,
            'negation_count': 0,
            'first_person_count': 0
        }
    
    text_lower = text.lower()
    words = text_lower.split()
    
    # 단어 수
    word_count = len(words)
    
    # 평균 단어 길이
    avg_word_length = np.mean([len(w) for w in words]) if words else 0.0
    
    # 문장 수 (간단한 추정)
    sentence_count = max(1, text.count('.') + text.count('!') + text.count('?'))
    
    # 부정어 카운트
    negation_words = ['no', 'not', 'never', "n't", 'nothing', 'nobody', 'nowhere', 
                     'neither', 'hardly', 'barely', 'scarcely', "don't", "didn't", 
                     "won't", "wouldn't", "can't", "couldn't", "shouldn't"]
    negation_count = sum(1 for word in words if word in negation_words)
    
    # 1인칭 대명사 카운트 (우울증 환자에서 높음)
    first_person = ['i', 'me', 'my', 'mine', 'myself']
    first_person_count = sum(1 for word in words if word in first_person)
    
    return {
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'sentence_count': sentence_count,
        'negation_count': negation_count,
        'first_person_count': first_person_count
    }


# =============================================================================
# Wav2Vec 특징 추출
# =============================================================================
def extract_wav2vec(y, sr):
    """Wav2Vec 2.0 특징 추출"""
    try:
        if len(y) < SR * 0.3:  # 0.3초 미만은 너무 짧음
            return None
        
        inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(DEVICE)
        
        with torch.no_grad():
            outputs = wav2vec_model(input_values)
            hidden_states = outputs.last_hidden_state  # [1, time_steps, 768]
        
        # 평균 풀링
        embedding = torch.mean(hidden_states, dim=1).squeeze().cpu().numpy()
        
        # 안전성 체크
        if not np.isfinite(embedding).all():
            return None
        
        return embedding
    except Exception as e:
        return None


# =============================================================================
# 대화 전처리 클래스
# =============================================================================
class UtterancePreprocessor:
    def __init__(self, base_path):
        self.base_path = base_path
    
    def normalize_question_type(self, q_type):
        """질문 유형 정규화 및 단순화"""
        q_type = q_type.lower().strip()
        q_type = ' '.join(q_type.split())  # 공백 정규화
        q_type = q_type.replace('/', ' / ')
        q_type = ' '.join(q_type.split())
        
        # 단순화
        if q_type in Q_TYPE_SIMPLIFICATION:
            return Q_TYPE_SIMPLIFICATION[q_type]
        return 'other'
    
    def process_transcript(self, pid):
        """CSV에서 발화 단위로 추출"""
        transcript_path = os.path.join(self.base_path, f"{pid}_P", f"{pid}_cleaned_transcript.csv")
        
        try:
            df = pd.read_csv(transcript_path, sep='\t')
            if df.shape[1] < 2:
                df = pd.read_csv(transcript_path, sep=',')
        except:
            return []
        
        # 컬럼명 정규화
        df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
        
        # question_label 처리
        if 'question_label' not in df.columns:
            df['question_label'] = 'other'
        df['question_label'] = df['question_label'].fillna('other')
        df['question_label'] = df['question_label'].replace('', 'other')
        
        # 발화 추출
        utterances = self._extract_utterances(df, pid)
        
        # 긴 발화 분할
        final_utterances = self._split_long_utterances(utterances)
        
        return final_utterances
    
    def _extract_utterances(self, df, pid):
        """발화 단위로 추출 (합치지 않음)"""
        utterances = []
        current_q_type = None
        first_ellie_found = False
        
        for idx, row in df.iterrows():
            speaker = str(row['speaker']).strip().lower()
            q_label = str(row['question_label']).strip().lower()
            
            # Ellie 발화
            if 'ellie' in speaker:
                first_ellie_found = True
                
                # 질문 유형 업데이트
                q_label = self.normalize_question_type(q_label)
                if q_label != 'other':
                    current_q_type = q_label
                else:
                    current_q_type = 'other'
            
            # Participant 발화
            elif 'participant' in speaker:
                if not first_ellie_found:
                    continue  # 첫 Ellie 발화 전 무시
                
                if current_q_type is None:
                    continue
                
                text = str(row['value'])
                start_time = row['start_time']
                stop_time = row['stop_time']
                duration = stop_time - start_time
                
                # 너무 짧은 발화 제거
                if duration < MIN_UTTERANCE_DURATION:
                    continue
                
                utterances.append({
                    'pid': pid,
                    'q_type': current_q_type,
                    'text': text,
                    'start': start_time,
                    'end': stop_time,
                    'duration': duration
                })
        
        return utterances
    
    def _split_long_utterances(self, utterances):
        """긴 발화를 15초 단위로 분할"""
        final_utterances = []
        
        for utt in utterances:
            duration = utt['duration']
            
            if duration <= MAX_UTTERANCE_DURATION:
                final_utterances.append(utt)
            else:
                # 분할
                num_splits = int(np.ceil(duration / MAX_UTTERANCE_DURATION))
                split_duration = duration / num_splits
                
                for i in range(num_splits):
                    split_start = utt['start'] + i * split_duration
                    split_end = min(split_start + split_duration, utt['end'])
                    
                    final_utterances.append({
                        'pid': utt['pid'],
                        'q_type': utt['q_type'],
                        'text': utt['text'],  # 텍스트는 동일하게 유지
                        'start': split_start,
                        'end': split_end,
                        'duration': split_end - split_start
                    })
        
        return final_utterances


# =============================================================================
# 오디오 품질 검증
# =============================================================================
def check_audio_quality(y, sr):
    """오디오 품질 검사"""
    issues = []
    
    # 1. 무음 비율 체크 (80% 이상 무음이면 문제)
    energy = librosa.feature.rms(y=y)[0]
    silence_ratio = np.sum(energy < 0.01) / len(energy)
    if silence_ratio > 0.8:
        issues.append(f"무음 비율 높음: {silence_ratio:.2%}")
    
    # 2. 클리핑 체크 (진폭이 0.99 이상인 비율)
    clipping_ratio = np.sum(np.abs(y) > 0.99) / len(y)
    if clipping_ratio > 0.01:
        issues.append(f"클리핑 발생: {clipping_ratio:.2%}")
    
    # 3. 너무 작은 볼륨
    max_amplitude = np.max(np.abs(y))
    if max_amplitude < 0.01:
        issues.append(f"볼륨 너무 작음: {max_amplitude:.4f}")
    
    return issues


# =============================================================================
# 메인 파이프라인
# =============================================================================
def run_preprocessing_pipeline():
    """전체 전처리 파이프라인 실행"""
    # 메타데이터 로드
    meta = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta['Participant_ID'] = meta['Participant_ID'].astype(str)
    meta = meta[~meta['Participant_ID'].isin(EXCEPTION_NUMBER)].reset_index(drop=True)
    
    preprocessor = UtterancePreprocessor(BASE_PATH)
    
    # 참가자별 데이터 저장
    dataset = {}
    
    print(f"\n{'='*70}")
    print(f"🚀 전처리 시작: {len(meta)}명")
    print(f"   - 오디오: Wav2Vec 2.0")
    print(f"   - 텍스트: {TEXT_MODEL_NAME}")
    print(f"{'='*70}\n")
    
    stats = {
        'processed': 0,
        'failed': 0,
        'total_utterances': 0,
        'low_quality': 0,
        'text_extraction_failed': 0,
        'q_type_counts': defaultdict(int)
    }
    
    quality_issues = []
    
    for idx, row in tqdm(meta.iterrows(), total=len(meta), desc="참가자 처리"):
        pid = str(row['Participant_ID'])
        label = int(row['Binary'])
        
        # 1. CSV에서 발화 추출
        utterances = preprocessor.process_transcript(pid)
        if not utterances:
            stats['failed'] += 1
            continue
        
        # 2. 오디오 로드 (전체 파일 한 번만)
        audio_path = os.path.join(BASE_PATH, f"{pid}_P", f"{pid}_AUDIO.wav")
        if not os.path.exists(audio_path):
            stats['failed'] += 1
            continue
        
        try:
            y_full, _ = librosa.load(audio_path, sr=SR)
            # 노이즈 제거
            y_full = nr.reduce_noise(y=y_full, sr=SR, stationary=True, prop_decrease=0.8)
        except Exception as e:
            stats['failed'] += 1
            continue
        
        # 3. 각 발화별 특징 추출
        processed_utterances = []
        
        for utt in utterances:
            # ==================== 오디오 특징 ====================
            # 오디오 추출
            start_sample = int(utt['start'] * SR)
            end_sample = int(utt['end'] * SR)
            
            if start_sample >= end_sample or end_sample > len(y_full):
                continue
            
            audio_segment = y_full[start_sample:end_sample]
            
            # 품질 검사
            issues = check_audio_quality(audio_segment, SR)
            if issues:
                quality_issues.append({
                    'pid': pid,
                    'duration': utt['duration'],
                    'issues': issues
                })
                stats['low_quality'] += 1
                # 심각한 문제 아니면 계속 진행
                if len(issues) > 2:  # 2개 이상 문제면 스킵
                    continue
            
            # Wav2Vec 특징 추출
            wav2vec_feat = extract_wav2vec(audio_segment, SR)
            if wav2vec_feat is None:
                continue
            
            # ==================== 텍스트 특징 ====================
            # BERT/MentalBERT 임베딩
            text_embedding = None
            if EXTRACT_TEXT_FEATURES:
                text_embedding = extract_text_embedding(utt['text'])
                if text_embedding is None:
                    stats['text_extraction_failed'] += 1
            
            # 언어학적 특징
            linguistic_feats = extract_linguistic_features(utt['text'])
            
            # TTR 계산
            ttr = get_ttr(utt['text'])
            
            # Q-type ID 변환
            q_type_id = Q_TYPE_MAPPING.get(utt['q_type'], Q_TYPE_MAPPING['other'])
            
            # ==================== 저장 ====================
            utterance_data = {
                # 오디오 특징
                'wav2vec': wav2vec_feat,  # [768]
                
                # 텍스트 특징
                'text_embedding': text_embedding,  # [768] or None
                'ttr': ttr,
                'word_count': linguistic_feats['word_count'],
                'avg_word_length': linguistic_feats['avg_word_length'],
                'sentence_count': linguistic_feats['sentence_count'],
                'negation_count': linguistic_feats['negation_count'],
                'first_person_count': linguistic_feats['first_person_count'],
                
                # 메타 정보
                'q_type': utt['q_type'],
                'q_type_id': q_type_id,
                'duration': utt['duration'],
                'text': utt['text']
            }
            
            processed_utterances.append(utterance_data)
            
            stats['total_utterances'] += 1
            stats['q_type_counts'][utt['q_type']] += 1
        
        if not processed_utterances:
            stats['failed'] += 1
            continue
        
        # 4. 참가자 데이터 저장
        dataset[pid] = {
            'label': label,
            'utterances': processed_utterances,
            'num_utterances': len(processed_utterances)
        }
        
        stats['processed'] += 1
    
    # 통계 출력
    print(f"\n{'='*70}")
    print(f"✅ 전처리 완료!")
    print(f"{'='*70}")
    print(f"처리 성공: {stats['processed']}명")
    print(f"처리 실패: {stats['failed']}명")
    print(f"총 발화: {stats['total_utterances']}개")
    print(f"품질 이슈: {stats['low_quality']}개 발화")
    print(f"텍스트 추출 실패: {stats['text_extraction_failed']}개 발화")
    
    print(f"\nQuestion Type 분포:")
    for q_type, count in sorted(stats['q_type_counts'].items(), key=lambda x: -x[1]):
        percentage = (count / stats['total_utterances']) * 100
        print(f"  {q_type:15s}: {count:5d}개 ({percentage:5.1f}%)")
    
    # 품질 이슈 샘플 출력
    if quality_issues:
        print(f"\n품질 이슈 샘플 (상위 10개):")
        for issue in quality_issues[:10]:
            print(f"  PID {issue['pid']}: {issue['duration']:.1f}초 - {', '.join(issue['issues'])}")
    
    return dataset


# =============================================================================
# 실행 및 저장
# =============================================================================
if __name__ == "__main__":
    # 전처리 실행
    dataset = run_preprocessing_pipeline()
    
    # 저장
    output_path = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")
    with open(output_path, 'wb') as f:
        pickle.dump(dataset, f)
    
    print(f"\n💾 데이터 저장 완료: {output_path}")
    
    # 샘플 데이터 확인
    sample_pid = list(dataset.keys())[0]
    sample = dataset[sample_pid]
    
    print(f"\n{'='*70}")
    print(f"📋 샘플 데이터 구조 (PID: {sample_pid})")
    print(f"{'='*70}")
    print(f"Label: {sample['label']}")
    print(f"Num Utterances: {sample['num_utterances']}")
    print(f"\n첫 번째 발화:")
    utt = sample['utterances'][0]
    print(f"  - Q-type: {utt['q_type']} (ID: {utt['q_type_id']})")
    print(f"  - Duration: {utt['duration']:.2f}초")
    print(f"  - Wav2Vec Shape: {utt['wav2vec'].shape}")
    
    if utt['text_embedding'] is not None:
        print(f"  - Text Embedding Shape: {utt['text_embedding'].shape}")
    else:
        print(f"  - Text Embedding: None")
    
    print(f"\n  텍스트 특징:")
    print(f"    - TTR: {utt['ttr']:.3f}")
    print(f"    - Word Count: {utt['word_count']}")
    print(f"    - Avg Word Length: {utt['avg_word_length']:.2f}")
    print(f"    - Sentence Count: {utt['sentence_count']}")
    print(f"    - Negation Count: {utt['negation_count']}")
    print(f"    - First Person Count: {utt['first_person_count']}")
    print(f"    - Text: {utt['text'][:80]}...")
    
    print(f"\n{'='*70}")
    print(f"✅ 모든 작업 완료!")
    print(f"{'='*70}")

c:\Users\Lenovo\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


⏳ Wav2Vec 2.0 모델 로딩 중... (Device: cuda)


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Wav2Vec 모델 로드 완료!
⏳ 텍스트 모델 로딩 중: roberta-base


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 텍스트 모델 로드 완료!

🚀 전처리 시작: 186명
   - 오디오: Wav2Vec 2.0
   - 텍스트: roberta-base



참가자 처리: 100%|██████████| 186/186 [37:29<00:00, 12.09s/it]



✅ 전처리 완료!
처리 성공: 186명
처리 실패: 0명
총 발화: 28580개
품질 이슈: 18248개 발화
텍스트 추출 실패: 0개 발화

Question Type 분포:
  background     : 11743개 ( 41.1%)
  casual         :  8313개 ( 29.1%)
  emotional      :  5607개 ( 19.6%)
  clinical       :  2917개 ( 10.2%)

품질 이슈 샘플 (상위 10개):
  PID 300: 3.1초 - 무음 비율 높음: 83.51%
  PID 300: 0.8초 - 무음 비율 높음: 100.00%
  PID 300: 1.5초 - 무음 비율 높음: 100.00%
  PID 300: 3.3초 - 무음 비율 높음: 100.00%
  PID 300: 0.6초 - 무음 비율 높음: 100.00%, 볼륨 너무 작음: 0.0087
  PID 300: 0.7초 - 무음 비율 높음: 100.00%
  PID 300: 0.8초 - 무음 비율 높음: 100.00%
  PID 300: 1.1초 - 무음 비율 높음: 94.12%
  PID 300: 1.5초 - 무음 비율 높음: 95.83%
  PID 300: 0.9초 - 무음 비율 높음: 100.00%

💾 데이터 저장 완료: D:\depression_dataset(DAIC-WOZ)\preprocessed_utterance_dataset_with_text.pkl

📋 샘플 데이터 구조 (PID: 300)
Label: 0
Num Utterances: 83

첫 번째 발화:
  - Q-type: casual (ID: 0)
  - Duration: 0.85초
  - Wav2Vec Shape: (768,)
  - Text Embedding Shape: (768,)

  텍스트 특징:
    - TTR: 1.000
    - Word Count: 1
    - Avg Word Length: 4.00
    - Sentence Count: 1
    - N

Adaptive Gating 

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
TEXT_EMBED_DIM = 768  # 텍스트 임베딩 (추후 추가 가능)
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1

# ⭐ Multimodal Fusion 설정
FUSION_METHOD = "adaptive_gate"  # "concat", "cross_attention", "adaptive_gate", "hierarchical"
USE_TEXT_FEATURES = False  # 현재는 오디오만, 추후 True로 변경

if USE_TEXT_FEATURES:
    INPUT_DIM = WAV2VEC_DIM + TEXT_EMBED_DIM + Q_TYPE_EMBED_DIM + TTR_DIM
else:
    INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# 불균형 처리 전략
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.45
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.08

# Threshold 동적 조정
DYNAMIC_THRESHOLD = True
MIN_RECALL_THRESHOLD = 0.70

EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# 개선된 Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    """Focal Loss: 어려운 샘플에 집중"""
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    """위치 정보 인코딩"""
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Multimodal Fusion Modules
# =============================================================================
class CrossModalAttention(nn.Module):
    """
    오디오와 텍스트 간 Cross-Attention
    Query: 한 modality, Key/Value: 다른 modality
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(CrossModalAttention, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key_value, mask=None):
        """
        Args:
            query: [batch, seq, d_model] - 주 modality
            key_value: [batch, seq, d_model] - 참조 modality
            mask: attention mask
        """
        # Cross-attention
        attn_out, attn_weights = self.cross_attn(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=mask
        )
        
        # Residual + Norm
        query = self.norm1(query + self.dropout(attn_out))
        
        # FFN
        ffn_out = self.ffn(query)
        query = self.norm2(query + ffn_out)
        
        return query, attn_weights


class AdaptiveGatingFusion(nn.Module):
    """
    Adaptive Gating: 각 modality의 기여도를 동적으로 조절
    G = sigmoid(W * [audio; text])
    fused = G * audio + (1-G) * text
    """
    def __init__(self, d_model, dropout=0.1):
        super(AdaptiveGatingFusion, self).__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, audio_features, text_features):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        
        Returns:
            fused: [batch, seq, d_model]
            gate_weights: [batch, seq, d_model] - 해석용
        """
        # Concatenate
        combined = torch.cat([audio_features, text_features], dim=-1)
        
        # Gate 계산
        gate_weights = self.gate(combined)
        
        # Adaptive fusion
        fused = gate_weights * audio_features + (1 - gate_weights) * text_features
        
        # Normalization
        fused = self.norm(fused)
        
        return fused, gate_weights


class HierarchicalFusion(nn.Module):
    """
    계층적 융합:
    1. 각 modality를 독립적으로 인코딩
    2. Cross-attention으로 상호작용
    3. Adaptive gating으로 최종 융합
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(HierarchicalFusion, self).__init__()
        # Cross-attention 양방향
        self.audio_to_text = CrossModalAttention(d_model, nhead, dropout)
        self.text_to_audio = CrossModalAttention(d_model, nhead, dropout)
        
        # Adaptive gating
        self.fusion = AdaptiveGatingFusion(d_model, dropout)
    
    def forward(self, audio_features, text_features, mask=None):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        """
        # Cross-modal attention
        audio_enhanced, _ = self.audio_to_text(audio_features, text_features, mask)
        text_enhanced, _ = self.text_to_audio(text_features, audio_features, mask)
        
        # Adaptive fusion
        fused, gate_weights = self.fusion(audio_enhanced, text_enhanced)
        
        return fused, gate_weights


# =============================================================================
# Multimodal Transformer 우울증 감지 모델
# =============================================================================
class MultimodalTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ):
        super(MultimodalTransformerModel, self).__init__()
        
        self.d_model = d_model
        self.fusion_method = fusion_method
        self.use_text_features = use_text_features
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # ⭐ Multimodal Fusion Module 선택
        if self.use_text_features:
            if fusion_method == "cross_attention":
                self.fusion_module = CrossModalAttention(d_model, nhead=4, dropout=dropout)
            elif fusion_method == "adaptive_gate":
                self.fusion_module = AdaptiveGatingFusion(d_model, dropout=dropout)
            elif fusion_method == "hierarchical":
                self.fusion_module = HierarchicalFusion(d_model, nhead=4, dropout=dropout)
            # "concat"는 별도 모듈 없이 그냥 결합
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier (개선된 구조)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list, batch_text_embed=None):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
            batch_text_embed: [total_utterances, 768] (optional)
        """
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        # ⭐ Multimodal 결합
        if self.use_text_features and batch_text_embed is not None:
            combined_features = torch.cat([
                batch_wav2vec,
                batch_text_embed,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        else:
            # Audio-only (기존)
            combined_features = torch.cat([
                batch_wav2vec,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Input Projection
        x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# PyTorch Dataset
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


# =============================================================================
# Collate Function
# =============================================================================
def collate_fn(batch):
    """배치 내 참가자들의 발화 수가 다르므로 동적 처리"""
    
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    # 특징 추출
    batch_wav2vec = []
    batch_ttrs = []
    batch_q_type_ids = []
    batch_text_embed = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        # 텍스트 임베딩 (있으면)
        if 'text_embedding' in utt and utt['text_embedding'] is not None:
            batch_text_embed.append(utt['text_embedding'])
    
    # Tensor 변환
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttrs = torch.FloatTensor(batch_ttrs)
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    result = {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }
    
    # 텍스트 임베딩 추가 (있으면)
    if batch_text_embed:
        result['batch_text_embed'] = torch.FloatTensor(np.array(batch_text_embed))
    
    return result


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    """전처리된 데이터 로드 및 Train/Val/Test 분할"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 기준 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # 전처리된 데이터에 있는 것만
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    """모델 평가 - 동적 threshold 지원"""
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            # 텍스트 임베딩 (있으면)
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    # 기본 메트릭
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # 동적 threshold 탐색
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(
            all_labels, 
            all_probs,
            min_recall=MIN_RECALL_THRESHOLD
        )
        return {
            'loss': avg_loss,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'labels': all_labels,
            'probs': all_probs,
            'preds': all_preds,
            'best_threshold': best_threshold,
            'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    """F1 최대화하는 threshold 찾기"""
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': thresh
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Training Loop
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """모델 학습"""
    
    # Loss function
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(
            alpha=FOCAL_ALPHA,
            gamma=FOCAL_GAMMA,
            label_smoothing=LABEL_SMOOTHING
        )
        print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    else:
        criterion = nn.BCEWithLogitsLoss()
        print(f"📍 Using BCE Loss")
    
    # Optimizer & Scheduler
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=warmup_epochs
    )
    
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=1e-6
    )
    
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )
    
    # Early stopping
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    # 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작 - Multimodal Fusion: {FUSION_METHOD}")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate(
            model, 
            val_loader, 
            criterion, 
            threshold=0.5,
            return_all_thresholds=True
        )
        
        # 최적 threshold 사용
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(val_results['f1'])
            history['val_precision'].append(val_results['precision'])
            history['val_recall'].append(val_results['recall'])
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        # Specificity
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss:  {avg_train_loss:.4f}")
        print(f"  Val Loss:    {val_results['loss']:.4f}")
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            print(f"  Optimal Threshold: {best_thresh:.3f}")
        
        print(f"  Val F1:      {optimal_f1:.4f}")
        print(f"  Val Prec:    {optimal_prec:.4f}")
        print(f"  Val Recall:  {optimal_rec:.4f}")
        print(f"  Val Spec:    {specificity:.4f}  ← 정상 구별")
        print(f"  LR:          {optimizer.param_groups[0]['lr']:.6f}")
        
        # Early Stopping
        is_balanced = (
            optimal_prec > 0.25 and
            optimal_rec > 0.60 and
            specificity > 0.25
        )
        
        if optimal_f1 > best_val_f1 and is_balanced:
            best_val_f1 = optimal_f1
            best_val_precision = optimal_prec
            best_val_recall = optimal_rec
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': best_val_f1,
                'val_precision': best_val_precision,
                'val_recall': best_val_recall,
                'val_specificity': specificity,
                'optimal_threshold': best_thresh,
                'history': history,
                'fusion_method': FUSION_METHOD
            }, os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            if not is_balanced:
                print(f"  ⚠️  Balanced 조건 미충족")
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping triggered!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history, fusion_method):
    """학습 곡선 시각화"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training & Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('Validation F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Val Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Validation Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Val Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Validation Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, f'training_history_multimodal_{fusion_method}.png'))
    plt.close()


def plot_confusion_matrix(labels, preds, title, fusion_method):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(labels, preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, f'{title.lower().replace(" ", "_")}_multimodal_{fusion_method}.png'))
    plt.close()


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = MultimodalTransformerModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료 - Multimodal Transformer")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Fusion Method: {FUSION_METHOD}")
    print(f"Use Text Features: {USE_TEXT_FEATURES}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # 학습 곡선 시각화
    plot_training_history(history, FUSION_METHOD)
    print(f"\n✅ 학습 곡선 저장")
    
    # 최적 모델 로드 및 평가
    model_path = os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt')
    
    if os.path.exists(model_path):
        print(f"\n{'='*70}")
        print(f"📥 최적 모델 로드 중...")
        print(f"{'='*70}")
        
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if 'optimal_threshold' in checkpoint:
            saved_threshold = checkpoint['optimal_threshold']
            print(f"💡 모델에 저장된 최적 threshold: {saved_threshold:.3f}")
        else:
            saved_threshold = 0.5
        
        # Validation 최종 확인
        val_results = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), 
                              threshold=saved_threshold, return_all_thresholds=True)
        
        if 'best_threshold' in val_results:
            best_threshold = val_results['best_threshold']
            best_met = val_results['best_metrics']
            print(f"💡 최종 검증 threshold: {best_threshold:.3f}")
            print(f"   → F1: {best_met['f1']:.4f}, Prec: {best_met['precision']:.4f}, Rec: {best_met['recall']:.4f}")
        else:
            best_threshold = saved_threshold
        
        # Test 평가
        print(f"\n{'='*70}")
        print(f"📊 최종 Test Set 평가 (Threshold: {best_threshold:.3f})")
        print(f"{'='*70}\n")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"Test Loss:      {test_results['loss']:.4f}")
        print(f"Test F1:        {test_results['f1']:.4f}")
        print(f"Test Precision: {test_results['precision']:.4f}")
        print(f"Test Recall:    {test_results['recall']:.4f}")
        
        # Confusion Matrix
        plot_confusion_matrix(test_results['labels'], test_results['preds'], 
                            'Test Confusion Matrix', FUSION_METHOD)
        
        # Classification Report
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(test_results['labels'], test_results['preds'],
                                    target_names=['Normal', 'Depression']))
        
        print(f"\n✅ 모든 학습 및 평가 완료!")
    else:
        print(f"\n⚠️  경고: 최적 모델이 저장되지 않았습니다.")

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료 - Multimodal Transformer
총 파라미터 수: 1,828,769
Fusion Method: adaptive_gate
Use Text Features: False
Device: cuda

📍 Using Focal Loss (alpha=0.45, gamma=1.5)

🚀 학습 시작 - Multimodal Fusion: adaptive_gate



Epoch 1/50: 100%|██████████| 14/14 [00:00<00:00, 15.19it/s, loss=0.126]



Epoch 1/50
  Train Loss:  0.1267
  Val Loss:    0.1264
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000028
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (1/15)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 37.95it/s, loss=0.117]



Epoch 2/50
  Train Loss:  0.1199
  Val Loss:    0.1148
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000046
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (2/15)
----------------------------------------------------------------------


...


Epoch 48/50
  Train Loss:  0.0983
  Val Loss:    0.1013
  Optimal Threshold: 0.360
  Val F1:      0.7143
  Val Prec:    0.6250
  Val Recall:  0.8333
  Val Spec:    0.7143  ← 정상 구별
  LR:          0.000001
  ⏳ No improvement (4/15)
----------------------------------------------------------------------


Epoch 49/50: 100%|██████████| 14/14 [00:00<00:00, 38.72it/s, loss=0.145] 



Epoch 49/50
  Train Loss:  0.1022
  Val Loss:    0.1013
  Optimal Threshold: 0.360
  Val F1:      0.7143
  Val Prec:    0.6250
  Val Recall:  0.8333
  Val Spec:    0.7143  ← 정상 구별
  LR:          0.000001
  ⏳ No improvement (5/15)
----------------------------------------------------------------------


Epoch 50/50: 100%|██████████| 14/14 [00:00<00:00, 38.47it/s, loss=0.0866]



Epoch 50/50
  Train Loss:  0.0955
  Val Loss:    0.1012
  Optimal Threshold: 0.360
  Val F1:      0.7143
  Val Prec:    0.6250
  Val Recall:  0.8333
  Val Spec:    0.7143  ← 정상 구별
  LR:          0.000001
  ⏳ No improvement (6/15)
----------------------------------------------------------------------

✅ 학습 곡선 저장

📥 최적 모델 로드 중...
💡 모델에 저장된 최적 threshold: 0.360
💡 최종 검증 threshold: 0.360
   → F1: 0.7333, Prec: 0.6111, Rec: 0.9167

📊 최종 Test Set 평가 (Threshold: 0.360)

Test Loss:      0.6017
Test F1:        0.6047
Test Precision: 0.4483
Test Recall:    0.9286

분류 보고서:
              precision    recall  f1-score   support

      Normal       0.94      0.50      0.65        32
  Depression       0.45      0.93      0.60        14

    accuracy                           0.63        46
   macro avg       0.69      0.71      0.63        46
weighted avg       0.79      0.63      0.64        46


✅ 모든 학습 및 평가 완료!


cross modal attention

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
TEXT_EMBED_DIM = 768  # 텍스트 임베딩 (추후 추가 가능)
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1

# ⭐ Multimodal Fusion 설정
FUSION_METHOD = "adaptive_gate"  # "concat", "cross_attention", "adaptive_gate", "hierarchical"
USE_TEXT_FEATURES = False  # 현재는 오디오만, 추후 True로 변경

if USE_TEXT_FEATURES:
    INPUT_DIM = WAV2VEC_DIM + TEXT_EMBED_DIM + Q_TYPE_EMBED_DIM + TTR_DIM
else:
    INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# 불균형 처리 전략
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.45
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.08

# Threshold 동적 조정
DYNAMIC_THRESHOLD = True
MIN_RECALL_THRESHOLD = 0.70

EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# 개선된 Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    """Focal Loss: 어려운 샘플에 집중"""
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    """위치 정보 인코딩"""
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Multimodal Fusion Modules
# =============================================================================
class CrossModalAttention(nn.Module):
    """
    오디오와 텍스트 간 Cross-Attention
    Query: 한 modality, Key/Value: 다른 modality
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(CrossModalAttention, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key_value, mask=None):
        """
        Args:
            query: [batch, seq, d_model] - 주 modality
            key_value: [batch, seq, d_model] - 참조 modality
            mask: attention mask
        """
        # Cross-attention
        attn_out, attn_weights = self.cross_attn(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=mask
        )
        
        # Residual + Norm
        query = self.norm1(query + self.dropout(attn_out))
        
        # FFN
        ffn_out = self.ffn(query)
        query = self.norm2(query + ffn_out)
        
        return query, attn_weights


class AdaptiveGatingFusion(nn.Module):
    """
    Adaptive Gating: 각 modality의 기여도를 동적으로 조절
    G = sigmoid(W * [audio; text])
    fused = G * audio + (1-G) * text
    """
    def __init__(self, d_model, dropout=0.1):
        super(AdaptiveGatingFusion, self).__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, audio_features, text_features):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        
        Returns:
            fused: [batch, seq, d_model]
            gate_weights: [batch, seq, d_model] - 해석용
        """
        # Concatenate
        combined = torch.cat([audio_features, text_features], dim=-1)
        
        # Gate 계산
        gate_weights = self.gate(combined)
        
        # Adaptive fusion
        fused = gate_weights * audio_features + (1 - gate_weights) * text_features
        
        # Normalization
        fused = self.norm(fused)
        
        return fused, gate_weights


class HierarchicalFusion(nn.Module):
    """
    계층적 융합:
    1. 각 modality를 독립적으로 인코딩
    2. Cross-attention으로 상호작용
    3. Adaptive gating으로 최종 융합
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(HierarchicalFusion, self).__init__()
        # Cross-attention 양방향
        self.audio_to_text = CrossModalAttention(d_model, nhead, dropout)
        self.text_to_audio = CrossModalAttention(d_model, nhead, dropout)
        
        # Adaptive gating
        self.fusion = AdaptiveGatingFusion(d_model, dropout)
    
    def forward(self, audio_features, text_features, mask=None):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        """
        # Cross-modal attention
        audio_enhanced, _ = self.audio_to_text(audio_features, text_features, mask)
        text_enhanced, _ = self.text_to_audio(text_features, audio_features, mask)
        
        # Adaptive fusion
        fused, gate_weights = self.fusion(audio_enhanced, text_enhanced)
        
        return fused, gate_weights


# =============================================================================
# Multimodal Transformer 우울증 감지 모델
# =============================================================================
class MultimodalTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ):
        super(MultimodalTransformerModel, self).__init__()
        
        self.d_model = d_model
        self.fusion_method = fusion_method
        self.use_text_features = use_text_features
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # ⭐ Multimodal Fusion Module 선택
        if self.use_text_features:
            if fusion_method == "cross_attention":
                self.fusion_module = CrossModalAttention(d_model, nhead=4, dropout=dropout)
            elif fusion_method == "adaptive_gate":
                self.fusion_module = AdaptiveGatingFusion(d_model, dropout=dropout)
            elif fusion_method == "hierarchical":
                self.fusion_module = HierarchicalFusion(d_model, nhead=4, dropout=dropout)
            # "concat"는 별도 모듈 없이 그냥 결합
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier (개선된 구조)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list, batch_text_embed=None):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
            batch_text_embed: [total_utterances, 768] (optional)
        """
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        # ⭐ Multimodal 결합
        if self.use_text_features and batch_text_embed is not None:
            combined_features = torch.cat([
                batch_wav2vec,
                batch_text_embed,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        else:
            # Audio-only (기존)
            combined_features = torch.cat([
                batch_wav2vec,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Input Projection
        x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# PyTorch Dataset
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


# =============================================================================
# Collate Function
# =============================================================================
def collate_fn(batch):
    """배치 내 참가자들의 발화 수가 다르므로 동적 처리"""
    
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    # 특징 추출
    batch_wav2vec = []
    batch_ttrs = []
    batch_q_type_ids = []
    batch_text_embed = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        # 텍스트 임베딩 (있으면)
        if 'text_embedding' in utt and utt['text_embedding'] is not None:
            batch_text_embed.append(utt['text_embedding'])
    
    # Tensor 변환
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttrs = torch.FloatTensor(batch_ttrs)
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    result = {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }
    
    # 텍스트 임베딩 추가 (있으면)
    if batch_text_embed:
        result['batch_text_embed'] = torch.FloatTensor(np.array(batch_text_embed))
    
    return result


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    """전처리된 데이터 로드 및 Train/Val/Test 분할"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 기준 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # 전처리된 데이터에 있는 것만
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    """모델 평가 - 동적 threshold 지원"""
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            # 텍스트 임베딩 (있으면)
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    # 기본 메트릭
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # 동적 threshold 탐색
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(
            all_labels, 
            all_probs,
            min_recall=MIN_RECALL_THRESHOLD
        )
        return {
            'loss': avg_loss,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'labels': all_labels,
            'probs': all_probs,
            'preds': all_preds,
            'best_threshold': best_threshold,
            'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    """F1 최대화하는 threshold 찾기"""
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': thresh
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Training Loop
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """모델 학습"""
    
    # Loss function
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(
            alpha=FOCAL_ALPHA,
            gamma=FOCAL_GAMMA,
            label_smoothing=LABEL_SMOOTHING
        )
        print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    else:
        criterion = nn.BCEWithLogitsLoss()
        print(f"📍 Using BCE Loss")
    
    # Optimizer & Scheduler
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=warmup_epochs
    )
    
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=1e-6
    )
    
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )
    
    # Early stopping
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    # 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작 - Multimodal Fusion: {FUSION_METHOD}")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate(
            model, 
            val_loader, 
            criterion, 
            threshold=0.5,
            return_all_thresholds=True
        )
        
        # 최적 threshold 사용
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(val_results['f1'])
            history['val_precision'].append(val_results['precision'])
            history['val_recall'].append(val_results['recall'])
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        # Specificity
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss:  {avg_train_loss:.4f}")
        print(f"  Val Loss:    {val_results['loss']:.4f}")
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            print(f"  Optimal Threshold: {best_thresh:.3f}")
        
        print(f"  Val F1:      {optimal_f1:.4f}")
        print(f"  Val Prec:    {optimal_prec:.4f}")
        print(f"  Val Recall:  {optimal_rec:.4f}")
        print(f"  Val Spec:    {specificity:.4f}  ← 정상 구별")
        print(f"  LR:          {optimizer.param_groups[0]['lr']:.6f}")
        
        # Early Stopping
        is_balanced = (
            optimal_prec > 0.25 and
            optimal_rec > 0.60 and
            specificity > 0.25
        )
        
        if optimal_f1 > best_val_f1 and is_balanced:
            best_val_f1 = optimal_f1
            best_val_precision = optimal_prec
            best_val_recall = optimal_rec
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': best_val_f1,
                'val_precision': best_val_precision,
                'val_recall': best_val_recall,
                'val_specificity': specificity,
                'optimal_threshold': best_thresh,
                'history': history,
                'fusion_method': FUSION_METHOD
            }, os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            if not is_balanced:
                print(f"  ⚠️  Balanced 조건 미충족")
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping triggered!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history, fusion_method):
    """학습 곡선 시각화"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training & Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('Validation F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Val Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Validation Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Val Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Validation Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, f'training_history_multimodal_{fusion_method}.png'))
    plt.close()


def plot_confusion_matrix(labels, preds, title, fusion_method):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(labels, preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, f'{title.lower().replace(" ", "_")}_multimodal_{fusion_method}.png'))
    plt.close()


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = MultimodalTransformerModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료 - Multimodal Transformer")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Fusion Method: {FUSION_METHOD}")
    print(f"Use Text Features: {USE_TEXT_FEATURES}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # 학습 곡선 시각화
    plot_training_history(history, FUSION_METHOD)
    print(f"\n✅ 학습 곡선 저장")
    
    # 최적 모델 로드 및 평가
    model_path = os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt')
    
    if os.path.exists(model_path):
        print(f"\n{'='*70}")
        print(f"📥 최적 모델 로드 중...")
        print(f"{'='*70}")
        
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if 'optimal_threshold' in checkpoint:
            saved_threshold = checkpoint['optimal_threshold']
            print(f"💡 모델에 저장된 최적 threshold: {saved_threshold:.3f}")
        else:
            saved_threshold = 0.5
        
        # Validation 최종 확인
        val_results = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), 
                              threshold=saved_threshold, return_all_thresholds=True)
        
        if 'best_threshold' in val_results:
            best_threshold = val_results['best_threshold']
            best_met = val_results['best_metrics']
            print(f"💡 최종 검증 threshold: {best_threshold:.3f}")
            print(f"   → F1: {best_met['f1']:.4f}, Prec: {best_met['precision']:.4f}, Rec: {best_met['recall']:.4f}")
        else:
            best_threshold = saved_threshold
        
        # Test 평가
        print(f"\n{'='*70}")
        print(f"📊 최종 Test Set 평가 (Threshold: {best_threshold:.3f})")
        print(f"{'='*70}\n")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"Test Loss:      {test_results['loss']:.4f}")
        print(f"Test F1:        {test_results['f1']:.4f}")
        print(f"Test Precision: {test_results['precision']:.4f}")
        print(f"Test Recall:    {test_results['recall']:.4f}")
        
        # Confusion Matrix
        plot_confusion_matrix(test_results['labels'], test_results['preds'], 
                            'Test Confusion Matrix', FUSION_METHOD)
        
        # Classification Report
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(test_results['labels'], test_results['preds'],
                                    target_names=['Normal', 'Depression']))
        
        print(f"\n✅ 모든 학습 및 평가 완료!")
    else:
        print(f"\n⚠️  경고: 최적 모델이 저장되지 않았습니다.")

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료 - Multimodal Transformer
총 파라미터 수: 1,828,769
Fusion Method: adaptive_gate
Use Text Features: False
Device: cuda

📍 Using Focal Loss (alpha=0.45, gamma=1.5)

🚀 학습 시작 - Multimodal Fusion: adaptive_gate



Epoch 1/50: 100%|██████████| 14/14 [00:00<00:00, 24.28it/s, loss=0.121]



Epoch 1/50
  Train Loss:  0.1272
  Val Loss:    0.1201
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000028
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (1/15)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 38.52it/s, loss=0.105]



Epoch 2/50
  Train Loss:  0.1198
  Val Loss:    0.1120
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000046
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (2/15)
----------------------------------------------------------------------


...


Epoch 36/50
  Train Loss:  0.1016
  Val Loss:    0.1061
  Optimal Threshold: 0.340
  Val F1:      0.6000
  Val Prec:    0.4286
  Val Recall:  1.0000
  Val Spec:    0.2381  ← 정상 구별
  LR:          0.000023
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 37/50: 100%|██████████| 14/14 [00:00<00:00, 36.93it/s, loss=0.0821]



Epoch 37/50
  Train Loss:  0.1023
  Val Loss:    0.1057
  Optimal Threshold: 0.340
  Val F1:      0.6154
  Val Prec:    0.4444
  Val Recall:  1.0000
  Val Spec:    0.2857  ← 정상 구별
  LR:          0.000020
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 38/50: 100%|██████████| 14/14 [00:00<00:00, 38.92it/s, loss=0.105] 



Epoch 38/50
  Train Loss:  0.1037
  Val Loss:    0.1049
  Optimal Threshold: 0.340
  Val F1:      0.6154
  Val Prec:    0.4444
  Val Recall:  1.0000
  Val Spec:    0.2857  ← 정상 구별
  LR:          0.000017
  ⏳ No improvement (15/15)

⚠️  Early stopping triggered!

✅ 학습 곡선 저장

📥 최적 모델 로드 중...
💡 모델에 저장된 최적 threshold: 0.380
💡 최종 검증 threshold: 0.380
   → F1: 0.6316, Prec: 0.4615, Rec: 1.0000

📊 최종 Test Set 평가 (Threshold: 0.380)

Test Loss:      0.6201
Test F1:        0.5385
Test Precision: 0.3684
Test Recall:    1.0000

분류 보고서:
              precision    recall  f1-score   support

      Normal       1.00      0.25      0.40        32
  Depression       0.37      1.00      0.54        14

    accuracy                           0.48        46
   macro avg       0.68      0.62      0.47        46
weighted avg       0.81      0.48      0.44        46


✅ 모든 학습 및 평가 완료!


concat

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
TEXT_EMBED_DIM = 768  # 텍스트 임베딩 (추후 추가 가능)
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1

# ⭐ Multimodal Fusion 설정
FUSION_METHOD = "concat"  # "concat", "cross_attention", "adaptive_gate", "hierarchical"
USE_TEXT_FEATURES = False  # 현재는 오디오만, 추후 True로 변경

if USE_TEXT_FEATURES:
    INPUT_DIM = WAV2VEC_DIM + TEXT_EMBED_DIM + Q_TYPE_EMBED_DIM + TTR_DIM
else:
    INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# 불균형 처리 전략
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.45
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.08

# Threshold 동적 조정
DYNAMIC_THRESHOLD = True
MIN_RECALL_THRESHOLD = 0.70

EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# 개선된 Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    """Focal Loss: 어려운 샘플에 집중"""
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    """위치 정보 인코딩"""
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Multimodal Fusion Modules
# =============================================================================
class CrossModalAttention(nn.Module):
    """
    오디오와 텍스트 간 Cross-Attention
    Query: 한 modality, Key/Value: 다른 modality
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(CrossModalAttention, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key_value, mask=None):
        """
        Args:
            query: [batch, seq, d_model] - 주 modality
            key_value: [batch, seq, d_model] - 참조 modality
            mask: attention mask
        """
        # Cross-attention
        attn_out, attn_weights = self.cross_attn(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=mask
        )
        
        # Residual + Norm
        query = self.norm1(query + self.dropout(attn_out))
        
        # FFN
        ffn_out = self.ffn(query)
        query = self.norm2(query + ffn_out)
        
        return query, attn_weights


class AdaptiveGatingFusion(nn.Module):
    """
    Adaptive Gating: 각 modality의 기여도를 동적으로 조절
    G = sigmoid(W * [audio; text])
    fused = G * audio + (1-G) * text
    """
    def __init__(self, d_model, dropout=0.1):
        super(AdaptiveGatingFusion, self).__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, audio_features, text_features):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        
        Returns:
            fused: [batch, seq, d_model]
            gate_weights: [batch, seq, d_model] - 해석용
        """
        # Concatenate
        combined = torch.cat([audio_features, text_features], dim=-1)
        
        # Gate 계산
        gate_weights = self.gate(combined)
        
        # Adaptive fusion
        fused = gate_weights * audio_features + (1 - gate_weights) * text_features
        
        # Normalization
        fused = self.norm(fused)
        
        return fused, gate_weights


class HierarchicalFusion(nn.Module):
    """
    계층적 융합:
    1. 각 modality를 독립적으로 인코딩
    2. Cross-attention으로 상호작용
    3. Adaptive gating으로 최종 융합
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(HierarchicalFusion, self).__init__()
        # Cross-attention 양방향
        self.audio_to_text = CrossModalAttention(d_model, nhead, dropout)
        self.text_to_audio = CrossModalAttention(d_model, nhead, dropout)
        
        # Adaptive gating
        self.fusion = AdaptiveGatingFusion(d_model, dropout)
    
    def forward(self, audio_features, text_features, mask=None):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        """
        # Cross-modal attention
        audio_enhanced, _ = self.audio_to_text(audio_features, text_features, mask)
        text_enhanced, _ = self.text_to_audio(text_features, audio_features, mask)
        
        # Adaptive fusion
        fused, gate_weights = self.fusion(audio_enhanced, text_enhanced)
        
        return fused, gate_weights


# =============================================================================
# Multimodal Transformer 우울증 감지 모델
# =============================================================================
class MultimodalTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ):
        super(MultimodalTransformerModel, self).__init__()
        
        self.d_model = d_model
        self.fusion_method = fusion_method
        self.use_text_features = use_text_features
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # ⭐ Multimodal Fusion Module 선택
        if self.use_text_features:
            if fusion_method == "cross_attention":
                self.fusion_module = CrossModalAttention(d_model, nhead=4, dropout=dropout)
            elif fusion_method == "adaptive_gate":
                self.fusion_module = AdaptiveGatingFusion(d_model, dropout=dropout)
            elif fusion_method == "hierarchical":
                self.fusion_module = HierarchicalFusion(d_model, nhead=4, dropout=dropout)
            # "concat"는 별도 모듈 없이 그냥 결합
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier (개선된 구조)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list, batch_text_embed=None):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
            batch_text_embed: [total_utterances, 768] (optional)
        """
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        # ⭐ Multimodal 결합
        if self.use_text_features and batch_text_embed is not None:
            combined_features = torch.cat([
                batch_wav2vec,
                batch_text_embed,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        else:
            # Audio-only (기존)
            combined_features = torch.cat([
                batch_wav2vec,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Input Projection
        x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# PyTorch Dataset
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


# =============================================================================
# Collate Function
# =============================================================================
def collate_fn(batch):
    """배치 내 참가자들의 발화 수가 다르므로 동적 처리"""
    
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    # 특징 추출
    batch_wav2vec = []
    batch_ttrs = []
    batch_q_type_ids = []
    batch_text_embed = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        # 텍스트 임베딩 (있으면)
        if 'text_embedding' in utt and utt['text_embedding'] is not None:
            batch_text_embed.append(utt['text_embedding'])
    
    # Tensor 변환
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttrs = torch.FloatTensor(batch_ttrs)
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    result = {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }
    
    # 텍스트 임베딩 추가 (있으면)
    if batch_text_embed:
        result['batch_text_embed'] = torch.FloatTensor(np.array(batch_text_embed))
    
    return result


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    """전처리된 데이터 로드 및 Train/Val/Test 분할"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 기준 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # 전처리된 데이터에 있는 것만
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    """모델 평가 - 동적 threshold 지원"""
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            # 텍스트 임베딩 (있으면)
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    # 기본 메트릭
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # 동적 threshold 탐색
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(
            all_labels, 
            all_probs,
            min_recall=MIN_RECALL_THRESHOLD
        )
        return {
            'loss': avg_loss,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'labels': all_labels,
            'probs': all_probs,
            'preds': all_preds,
            'best_threshold': best_threshold,
            'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    """F1 최대화하는 threshold 찾기"""
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': thresh
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Training Loop
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """모델 학습"""
    
    # Loss function
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(
            alpha=FOCAL_ALPHA,
            gamma=FOCAL_GAMMA,
            label_smoothing=LABEL_SMOOTHING
        )
        print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    else:
        criterion = nn.BCEWithLogitsLoss()
        print(f"📍 Using BCE Loss")
    
    # Optimizer & Scheduler
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=warmup_epochs
    )
    
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=1e-6
    )
    
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )
    
    # Early stopping
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    # 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작 - Multimodal Fusion: {FUSION_METHOD}")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate(
            model, 
            val_loader, 
            criterion, 
            threshold=0.5,
            return_all_thresholds=True
        )
        
        # 최적 threshold 사용
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(val_results['f1'])
            history['val_precision'].append(val_results['precision'])
            history['val_recall'].append(val_results['recall'])
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        # Specificity
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss:  {avg_train_loss:.4f}")
        print(f"  Val Loss:    {val_results['loss']:.4f}")
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            print(f"  Optimal Threshold: {best_thresh:.3f}")
        
        print(f"  Val F1:      {optimal_f1:.4f}")
        print(f"  Val Prec:    {optimal_prec:.4f}")
        print(f"  Val Recall:  {optimal_rec:.4f}")
        print(f"  Val Spec:    {specificity:.4f}  ← 정상 구별")
        print(f"  LR:          {optimizer.param_groups[0]['lr']:.6f}")
        
        # Early Stopping
        is_balanced = (
            optimal_prec > 0.25 and
            optimal_rec > 0.60 and
            specificity > 0.25
        )
        
        if optimal_f1 > best_val_f1 and is_balanced:
            best_val_f1 = optimal_f1
            best_val_precision = optimal_prec
            best_val_recall = optimal_rec
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': best_val_f1,
                'val_precision': best_val_precision,
                'val_recall': best_val_recall,
                'val_specificity': specificity,
                'optimal_threshold': best_thresh,
                'history': history,
                'fusion_method': FUSION_METHOD
            }, os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            if not is_balanced:
                print(f"  ⚠️  Balanced 조건 미충족")
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping triggered!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history, fusion_method):
    """학습 곡선 시각화"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training & Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('Validation F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Val Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Validation Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Val Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Validation Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, f'training_history_multimodal_{fusion_method}.png'))
    plt.close()


def plot_confusion_matrix(labels, preds, title, fusion_method):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(labels, preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, f'{title.lower().replace(" ", "_")}_multimodal_{fusion_method}.png'))
    plt.close()


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = MultimodalTransformerModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료 - Multimodal Transformer")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Fusion Method: {FUSION_METHOD}")
    print(f"Use Text Features: {USE_TEXT_FEATURES}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # 학습 곡선 시각화
    plot_training_history(history, FUSION_METHOD)
    print(f"\n✅ 학습 곡선 저장")
    
    # 최적 모델 로드 및 평가
    model_path = os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt')
    
    if os.path.exists(model_path):
        print(f"\n{'='*70}")
        print(f"📥 최적 모델 로드 중...")
        print(f"{'='*70}")
        
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if 'optimal_threshold' in checkpoint:
            saved_threshold = checkpoint['optimal_threshold']
            print(f"💡 모델에 저장된 최적 threshold: {saved_threshold:.3f}")
        else:
            saved_threshold = 0.5
        
        # Validation 최종 확인
        val_results = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), 
                              threshold=saved_threshold, return_all_thresholds=True)
        
        if 'best_threshold' in val_results:
            best_threshold = val_results['best_threshold']
            best_met = val_results['best_metrics']
            print(f"💡 최종 검증 threshold: {best_threshold:.3f}")
            print(f"   → F1: {best_met['f1']:.4f}, Prec: {best_met['precision']:.4f}, Rec: {best_met['recall']:.4f}")
        else:
            best_threshold = saved_threshold
        
        # Test 평가
        print(f"\n{'='*70}")
        print(f"📊 최종 Test Set 평가 (Threshold: {best_threshold:.3f})")
        print(f"{'='*70}\n")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"Test Loss:      {test_results['loss']:.4f}")
        print(f"Test F1:        {test_results['f1']:.4f}")
        print(f"Test Precision: {test_results['precision']:.4f}")
        print(f"Test Recall:    {test_results['recall']:.4f}")
        
        # Confusion Matrix
        plot_confusion_matrix(test_results['labels'], test_results['preds'], 
                            'Test Confusion Matrix', FUSION_METHOD)
        
        # Classification Report
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(test_results['labels'], test_results['preds'],
                                    target_names=['Normal', 'Depression']))
        
        print(f"\n✅ 모든 학습 및 평가 완료!")
    else:
        print(f"\n⚠️  경고: 최적 모델이 저장되지 않았습니다.")

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료 - Multimodal Transformer
총 파라미터 수: 1,828,769
Fusion Method: concat
Use Text Features: False
Device: cuda

📍 Using Focal Loss (alpha=0.45, gamma=1.5)

🚀 학습 시작 - Multimodal Fusion: concat



Epoch 1/50: 100%|██████████| 14/14 [00:00<00:00, 20.77it/s, loss=0.122]



Epoch 1/50
  Train Loss:  0.1198
  Val Loss:    0.1163
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000028
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (1/15)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 35.92it/s, loss=0.12] 



Epoch 2/50
  Train Loss:  0.1169
  Val Loss:    0.1105
  Optimal Threshold: 0.420
  Val F1:      0.5500
  Val Prec:    0.3929
  Val Recall:  0.9167
  Val Spec:    0.1905  ← 정상 구별
  LR:          0.000046
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (2/15)
----------------------------------------------------------------------


...


Epoch 38/50
  Train Loss:  0.0730
  Val Loss:    0.0913
  Optimal Threshold: 0.320
  Val F1:      0.7333
  Val Prec:    0.6111
  Val Recall:  0.9167
  Val Spec:    0.6667  ← 정상 구별
  LR:          0.000017
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 39/50: 100%|██████████| 14/14 [00:00<00:00, 38.09it/s, loss=0.166] 



Epoch 39/50
  Train Loss:  0.0824
  Val Loss:    0.0975
  Optimal Threshold: 0.280
  Val F1:      0.7333
  Val Prec:    0.6111
  Val Recall:  0.9167
  Val Spec:    0.6667  ← 정상 구별
  LR:          0.000015
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 40/50: 100%|██████████| 14/14 [00:00<00:00, 37.61it/s, loss=0.0399]



Epoch 40/50
  Train Loss:  0.0733
  Val Loss:    0.0983
  Optimal Threshold: 0.280
  Val F1:      0.7333
  Val Prec:    0.6111
  Val Recall:  0.9167
  Val Spec:    0.6667  ← 정상 구별
  LR:          0.000013
  ⏳ No improvement (15/15)

⚠️  Early stopping triggered!

✅ 학습 곡선 저장

📥 최적 모델 로드 중...
💡 모델에 저장된 최적 threshold: 0.360
💡 최종 검증 threshold: 0.360
   → F1: 0.7857, Prec: 0.6875, Rec: 0.9167

📊 최종 Test Set 평가 (Threshold: 0.360)

Test Loss:      0.5645
Test F1:        0.5556
Test Precision: 0.4545
Test Recall:    0.7143

분류 보고서:
              precision    recall  f1-score   support

      Normal       0.83      0.62      0.71        32
  Depression       0.45      0.71      0.56        14

    accuracy                           0.65        46
   macro avg       0.64      0.67      0.63        46
weighted avg       0.72      0.65      0.67        46


✅ 모든 학습 및 평가 완료!


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
TEXT_EMBED_DIM = 768  # 텍스트 임베딩 (추후 추가 가능)
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1

# ⭐ Multimodal Fusion 설정
FUSION_METHOD = "hierarchical"  # "concat", "cross_attention", "adaptive_gate", "hierarchical"
USE_TEXT_FEATURES = False  # 현재는 오디오만, 추후 True로 변경

if USE_TEXT_FEATURES:
    INPUT_DIM = WAV2VEC_DIM + TEXT_EMBED_DIM + Q_TYPE_EMBED_DIM + TTR_DIM
else:
    INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# 불균형 처리 전략
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.45
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.08

# Threshold 동적 조정
DYNAMIC_THRESHOLD = True
MIN_RECALL_THRESHOLD = 0.70

EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# 개선된 Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    """Focal Loss: 어려운 샘플에 집중"""
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    """위치 정보 인코딩"""
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Multimodal Fusion Modules
# =============================================================================
class CrossModalAttention(nn.Module):
    """
    오디오와 텍스트 간 Cross-Attention
    Query: 한 modality, Key/Value: 다른 modality
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(CrossModalAttention, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key_value, mask=None):
        """
        Args:
            query: [batch, seq, d_model] - 주 modality
            key_value: [batch, seq, d_model] - 참조 modality
            mask: attention mask
        """
        # Cross-attention
        attn_out, attn_weights = self.cross_attn(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=mask
        )
        
        # Residual + Norm
        query = self.norm1(query + self.dropout(attn_out))
        
        # FFN
        ffn_out = self.ffn(query)
        query = self.norm2(query + ffn_out)
        
        return query, attn_weights


class AdaptiveGatingFusion(nn.Module):
    """
    Adaptive Gating: 각 modality의 기여도를 동적으로 조절
    G = sigmoid(W * [audio; text])
    fused = G * audio + (1-G) * text
    """
    def __init__(self, d_model, dropout=0.1):
        super(AdaptiveGatingFusion, self).__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, audio_features, text_features):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        
        Returns:
            fused: [batch, seq, d_model]
            gate_weights: [batch, seq, d_model] - 해석용
        """
        # Concatenate
        combined = torch.cat([audio_features, text_features], dim=-1)
        
        # Gate 계산
        gate_weights = self.gate(combined)
        
        # Adaptive fusion
        fused = gate_weights * audio_features + (1 - gate_weights) * text_features
        
        # Normalization
        fused = self.norm(fused)
        
        return fused, gate_weights


class HierarchicalFusion(nn.Module):
    """
    계층적 융합:
    1. 각 modality를 독립적으로 인코딩
    2. Cross-attention으로 상호작용
    3. Adaptive gating으로 최종 융합
    """
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(HierarchicalFusion, self).__init__()
        # Cross-attention 양방향
        self.audio_to_text = CrossModalAttention(d_model, nhead, dropout)
        self.text_to_audio = CrossModalAttention(d_model, nhead, dropout)
        
        # Adaptive gating
        self.fusion = AdaptiveGatingFusion(d_model, dropout)
    
    def forward(self, audio_features, text_features, mask=None):
        """
        Args:
            audio_features: [batch, seq, d_model]
            text_features: [batch, seq, d_model]
        """
        # Cross-modal attention
        audio_enhanced, _ = self.audio_to_text(audio_features, text_features, mask)
        text_enhanced, _ = self.text_to_audio(text_features, audio_features, mask)
        
        # Adaptive fusion
        fused, gate_weights = self.fusion(audio_enhanced, text_enhanced)
        
        return fused, gate_weights


# =============================================================================
# Multimodal Transformer 우울증 감지 모델
# =============================================================================
class MultimodalTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ):
        super(MultimodalTransformerModel, self).__init__()
        
        self.d_model = d_model
        self.fusion_method = fusion_method
        self.use_text_features = use_text_features
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # ⭐ Multimodal Fusion Module 선택
        if self.use_text_features:
            if fusion_method == "cross_attention":
                self.fusion_module = CrossModalAttention(d_model, nhead=4, dropout=dropout)
            elif fusion_method == "adaptive_gate":
                self.fusion_module = AdaptiveGatingFusion(d_model, dropout=dropout)
            elif fusion_method == "hierarchical":
                self.fusion_module = HierarchicalFusion(d_model, nhead=4, dropout=dropout)
            # "concat"는 별도 모듈 없이 그냥 결합
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier (개선된 구조)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list, batch_text_embed=None):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
            batch_text_embed: [total_utterances, 768] (optional)
        """
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        # ⭐ Multimodal 결합
        if self.use_text_features and batch_text_embed is not None:
            combined_features = torch.cat([
                batch_wav2vec,
                batch_text_embed,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        else:
            # Audio-only (기존)
            combined_features = torch.cat([
                batch_wav2vec,
                q_type_embs,
                ttrs_expanded
            ], dim=1)
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Input Projection
        x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# PyTorch Dataset
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


# =============================================================================
# Collate Function
# =============================================================================
def collate_fn(batch):
    """배치 내 참가자들의 발화 수가 다르므로 동적 처리"""
    
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    # 특징 추출
    batch_wav2vec = []
    batch_ttrs = []
    batch_q_type_ids = []
    batch_text_embed = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        # 텍스트 임베딩 (있으면)
        if 'text_embedding' in utt and utt['text_embedding'] is not None:
            batch_text_embed.append(utt['text_embedding'])
    
    # Tensor 변환
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttrs = torch.FloatTensor(batch_ttrs)
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    result = {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }
    
    # 텍스트 임베딩 추가 (있으면)
    if batch_text_embed:
        result['batch_text_embed'] = torch.FloatTensor(np.array(batch_text_embed))
    
    return result


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    """전처리된 데이터 로드 및 Train/Val/Test 분할"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 기준 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # 전처리된 데이터에 있는 것만
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    """모델 평가 - 동적 threshold 지원"""
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            # 텍스트 임베딩 (있으면)
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    # 기본 메트릭
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # 동적 threshold 탐색
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(
            all_labels, 
            all_probs,
            min_recall=MIN_RECALL_THRESHOLD
        )
        return {
            'loss': avg_loss,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'labels': all_labels,
            'probs': all_probs,
            'preds': all_preds,
            'best_threshold': best_threshold,
            'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    """F1 최대화하는 threshold 찾기"""
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': thresh
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Training Loop
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """모델 학습"""
    
    # Loss function
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(
            alpha=FOCAL_ALPHA,
            gamma=FOCAL_GAMMA,
            label_smoothing=LABEL_SMOOTHING
        )
        print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    else:
        criterion = nn.BCEWithLogitsLoss()
        print(f"📍 Using BCE Loss")
    
    # Optimizer & Scheduler
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=warmup_epochs
    )
    
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=1e-6
    )
    
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )
    
    # Early stopping
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    # 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작 - Multimodal Fusion: {FUSION_METHOD}")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate(
            model, 
            val_loader, 
            criterion, 
            threshold=0.5,
            return_all_thresholds=True
        )
        
        # 최적 threshold 사용
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(val_results['f1'])
            history['val_precision'].append(val_results['precision'])
            history['val_recall'].append(val_results['recall'])
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        # Specificity
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss:  {avg_train_loss:.4f}")
        print(f"  Val Loss:    {val_results['loss']:.4f}")
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            print(f"  Optimal Threshold: {best_thresh:.3f}")
        
        print(f"  Val F1:      {optimal_f1:.4f}")
        print(f"  Val Prec:    {optimal_prec:.4f}")
        print(f"  Val Recall:  {optimal_rec:.4f}")
        print(f"  Val Spec:    {specificity:.4f}  ← 정상 구별")
        print(f"  LR:          {optimizer.param_groups[0]['lr']:.6f}")
        
        # ⭐ Early Stopping 조건 완화 (최소 저장 보장)
        is_balanced = (
            optimal_prec > 0.20 and  # 0.25 → 0.20
            optimal_rec > 0.55 and   # 0.60 → 0.55
            specificity > 0.20       # 0.25 → 0.20
        )
        
        # F1이 개선되거나, 첫 epoch이거나, balanced 조건 충족 시 저장
        should_save = (optimal_f1 > best_val_f1) or (epoch == 0) or (is_balanced and optimal_f1 > 0.5)
        
        if should_save:
            best_val_f1 = optimal_f1
            best_val_precision = optimal_prec
            best_val_recall = optimal_rec
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': best_val_f1,
                'val_precision': best_val_precision,
                'val_recall': best_val_recall,
                'val_specificity': specificity,
                'optimal_threshold': best_thresh,
                'history': history,
                'fusion_method': FUSION_METHOD
            }, os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            if not is_balanced:
                print(f"  ⚠️  Balanced 조건 미충족")
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping triggered!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history, fusion_method):
    """학습 곡선 시각화"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training & Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('Validation F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Val Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Validation Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Val Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Validation Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, f'training_history_multimodal_{fusion_method}.png'))
    plt.close()


def plot_confusion_matrix(labels, preds, title, fusion_method):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(labels, preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, f'{title.lower().replace(" ", "_")}_multimodal_{fusion_method}.png'))
    plt.close()


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = MultimodalTransformerModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료 - Multimodal Transformer")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Fusion Method: {FUSION_METHOD}")
    print(f"Use Text Features: {USE_TEXT_FEATURES}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # 학습 곡선 시각화
    plot_training_history(history, FUSION_METHOD)
    print(f"\n✅ 학습 곡선 저장")
    
    # 최적 모델 로드 및 평가
    model_path = os.path.join(BASE_PATH, f'best_multimodal_model_{FUSION_METHOD}.pt')
    
    if os.path.exists(model_path):
        print(f"\n{'='*70}")
        print(f"📥 최적 모델 로드 중...")
        print(f"{'='*70}")
        
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if 'optimal_threshold' in checkpoint:
            saved_threshold = checkpoint['optimal_threshold']
            print(f"💡 모델에 저장된 최적 threshold: {saved_threshold:.3f}")
        else:
            saved_threshold = 0.5
        
        # Validation 최종 확인
        val_results = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), 
                              threshold=saved_threshold, return_all_thresholds=True)
        
        if 'best_threshold' in val_results:
            best_threshold = val_results['best_threshold']
            best_met = val_results['best_metrics']
            print(f"💡 최종 검증 threshold: {best_threshold:.3f}")
            print(f"   → F1: {best_met['f1']:.4f}, Prec: {best_met['precision']:.4f}, Rec: {best_met['recall']:.4f}")
        else:
            best_threshold = saved_threshold
        
        # Test 평가
        print(f"\n{'='*70}")
        print(f"📊 최종 Test Set 평가 (Threshold: {best_threshold:.3f})")
        print(f"{'='*70}\n")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"Test Loss:      {test_results['loss']:.4f}")
        print(f"Test F1:        {test_results['f1']:.4f}")
        print(f"Test Precision: {test_results['precision']:.4f}")
        print(f"Test Recall:    {test_results['recall']:.4f}")
        
        # Confusion Matrix
        plot_confusion_matrix(test_results['labels'], test_results['preds'], 
                            'Test Confusion Matrix', FUSION_METHOD)
        
        # Classification Report
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(test_results['labels'], test_results['preds'],
                                    target_names=['Normal', 'Depression']))
        
        print(f"\n✅ 모든 학습 및 평가 완료!")
    else:
        print(f"\n⚠️  경고: 최적 모델이 저장되지 않았습니다.")

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료 - Multimodal Transformer
총 파라미터 수: 1,828,769
Fusion Method: hierarchical
Use Text Features: False
Device: cuda

📍 Using Focal Loss (alpha=0.45, gamma=1.5)

🚀 학습 시작 - Multimodal Fusion: hierarchical



Epoch 1/50: 100%|██████████| 14/14 [00:00<00:00, 24.86it/s, loss=0.121]



Epoch 1/50
  Train Loss:  0.1307
  Val Loss:    0.1294
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000028
  ✅ Best model saved! (F1: 0.5333)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 37.28it/s, loss=0.119]



Epoch 2/50
  Train Loss:  0.1234
  Val Loss:    0.1172
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000046
  ⚠️  Balanced 조건 미충족
  ⏳ No improvement (1/15)
----------------------------------------------------------------------


...


Epoch 48/50
  Train Loss:  0.0767
  Val Loss:    0.0948
  Optimal Threshold: 0.260
  Val F1:      0.7097
  Val Prec:    0.5789
  Val Recall:  0.9167
  Val Spec:    0.6190  ← 정상 구별
  LR:          0.000001
  ✅ Best model saved! (F1: 0.7097)
----------------------------------------------------------------------


Epoch 49/50: 100%|██████████| 14/14 [00:00<00:00, 37.76it/s, loss=0.0317]



Epoch 49/50
  Train Loss:  0.0798
  Val Loss:    0.0950
  Optimal Threshold: 0.260
  Val F1:      0.7097
  Val Prec:    0.5789
  Val Recall:  0.9167
  Val Spec:    0.6190  ← 정상 구별
  LR:          0.000001
  ✅ Best model saved! (F1: 0.7097)
----------------------------------------------------------------------


Epoch 50/50: 100%|██████████| 14/14 [00:00<00:00, 37.45it/s, loss=0.0453]



Epoch 50/50
  Train Loss:  0.0816
  Val Loss:    0.0948
  Optimal Threshold: 0.260
  Val F1:      0.7097
  Val Prec:    0.5789
  Val Recall:  0.9167
  Val Spec:    0.6190  ← 정상 구별
  LR:          0.000001
  ✅ Best model saved! (F1: 0.7097)
----------------------------------------------------------------------

✅ 학습 곡선 저장

📥 최적 모델 로드 중...
💡 모델에 저장된 최적 threshold: 0.260
💡 최종 검증 threshold: 0.260
   → F1: 0.7097, Prec: 0.5789, Rec: 0.9167

📊 최종 Test Set 평가 (Threshold: 0.260)

Test Loss:      0.5480
Test F1:        0.5789
Test Precision: 0.4583
Test Recall:    0.7857

분류 보고서:
              precision    recall  f1-score   support

      Normal       0.86      0.59      0.70        32
  Depression       0.46      0.79      0.58        14

    accuracy                           0.65        46
   macro avg       0.66      0.69      0.64        46
weighted avg       0.74      0.65      0.67        46


✅ 모든 학습 및 평가 완료!


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
# ⭐ 텍스트 특징 포함된 데이터셋 사용
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
TEXT_EMBED_DIM = 768  # MentalBERT/BERT 임베딩
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1
LINGUISTIC_FEATURES_DIM = 5  # word_count, avg_word_length, sentence_count, negation_count, first_person_count

# ⭐ Multimodal Fusion 설정
FUSION_METHOD = "hierarchical"  # "concat", "cross_attention", "adaptive_gate", "hierarchical"
USE_TEXT_FEATURES = True  # ⭐ 텍스트 사용!

if USE_TEXT_FEATURES:
    # Audio + Text + Q-type + TTR + Linguistic
    INPUT_DIM = WAV2VEC_DIM + TEXT_EMBED_DIM + Q_TYPE_EMBED_DIM + TTR_DIM + LINGUISTIC_FEATURES_DIM
    # 768 + 768 + 32 + 1 + 5 = 1574
else:
    INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM

# 모델 크기 (텍스트 추가로 복잡도 증가하므로 조정)
D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.4  # 0.3 → 0.4 (과적합 방지)

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 5e-5  # 1e-4 → 5e-5 (더 신중하게)
WEIGHT_DECAY = 1e-3   # 1e-4 → 1e-3 (정규화 강화)

# 불균형 처리
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.50  # 0.45 → 0.50 (균형)
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.10  # 0.08 → 0.10

# Threshold
DYNAMIC_THRESHOLD = True
MIN_RECALL_THRESHOLD = 0.70

EARLY_STOPPING_PATIENCE = 20  # 15 → 20
GRADIENT_CLIP = 0.5  # 1.0 → 0.5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Multimodal Fusion Modules
# =============================================================================
class CrossModalAttention(nn.Module):
    """오디오와 텍스트 간 Cross-Attention"""
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(CrossModalAttention, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key_value, mask=None):
        attn_out, attn_weights = self.cross_attn(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=mask
        )
        
        query = self.norm1(query + self.dropout(attn_out))
        ffn_out = self.ffn(query)
        query = self.norm2(query + ffn_out)
        
        return query, attn_weights


class AdaptiveGatingFusion(nn.Module):
    """Adaptive Gating: 동적 modality 가중치"""
    def __init__(self, d_model, dropout=0.1):
        super(AdaptiveGatingFusion, self).__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, audio_features, text_features):
        combined = torch.cat([audio_features, text_features], dim=-1)
        gate_weights = self.gate(combined)
        fused = gate_weights * audio_features + (1 - gate_weights) * text_features
        fused = self.norm(fused)
        
        return fused, gate_weights


class HierarchicalFusion(nn.Module):
    """계층적 융합: Cross-attention + Adaptive gating"""
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super(HierarchicalFusion, self).__init__()
        self.audio_to_text = CrossModalAttention(d_model, nhead, dropout)
        self.text_to_audio = CrossModalAttention(d_model, nhead, dropout)
        self.fusion = AdaptiveGatingFusion(d_model, dropout)
    
    def forward(self, audio_features, text_features, mask=None):
        audio_enhanced, _ = self.audio_to_text(audio_features, text_features, mask)
        text_enhanced, _ = self.text_to_audio(text_features, audio_features, mask)
        fused, gate_weights = self.fusion(audio_enhanced, text_enhanced)
        
        return fused, gate_weights


# =============================================================================
# ⭐ Modality-Specific Encoders
# =============================================================================
class ModalityEncoder(nn.Module):
    """각 Modality를 독립적으로 인코딩"""
    def __init__(self, input_dim, d_model, dropout=0.1):
        super(ModalityEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model)
        )
    
    def forward(self, x):
        return self.encoder(x)


# =============================================================================
# Multimodal Transformer 우울증 감지 모델
# =============================================================================
class MultimodalTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ):
        super(MultimodalTransformerModel, self).__init__()
        
        self.d_model = d_model
        self.fusion_method = fusion_method
        self.use_text_features = use_text_features
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # ⭐ Modality-Specific Encoders
        if self.use_text_features:
            # Audio modality encoder
            audio_input_dim = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM
            self.audio_encoder = ModalityEncoder(audio_input_dim, d_model, dropout)
            
            # Text modality encoder
            text_input_dim = TEXT_EMBED_DIM + LINGUISTIC_FEATURES_DIM
            self.text_encoder = ModalityEncoder(text_input_dim, d_model, dropout)
            
            # Fusion module
            if fusion_method == "cross_attention":
                self.fusion_module = CrossModalAttention(d_model, nhead=4, dropout=dropout)
            elif fusion_method == "adaptive_gate":
                self.fusion_module = AdaptiveGatingFusion(d_model, dropout=dropout)
            elif fusion_method == "hierarchical":
                self.fusion_module = HierarchicalFusion(d_model, nhead=4, dropout=dropout)
        else:
            # Audio-only
            self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list, 
                batch_text_embed=None, batch_linguistic=None):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
            batch_text_embed: [total_utterances, 768] (optional)
            batch_linguistic: [total_utterances, 5] (optional)
        """
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Question type embedding
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        # ⭐ Multimodal Processing
        if self.use_text_features and batch_text_embed is not None and batch_linguistic is not None:
            # Audio features
            audio_features = torch.cat([batch_wav2vec, q_type_embs, ttrs_expanded], dim=1)
            
            # Text features
            text_features = torch.cat([batch_text_embed, batch_linguistic], dim=1)
            
            # Encode separately
            audio_encoded = self.audio_encoder(audio_features)
            text_encoded = self.text_encoder(text_features)
            
            # 참가자별로 재구성
            audio_seqs = []
            text_seqs = []
            start_idx = 0
            
            for num_utts in num_utterances_list:
                end_idx = start_idx + num_utts
                audio_seqs.append(audio_encoded[start_idx:end_idx])
                text_seqs.append(text_encoded[start_idx:end_idx])
                start_idx = end_idx
            
            # 패딩
            max_num_utterances = max(num_utterances_list)
            padded_audio = []
            padded_text = []
            attention_masks = []
            
            for audio_seq, text_seq, num_utts in zip(audio_seqs, text_seqs, num_utterances_list):
                if num_utts < max_num_utterances:
                    padding = torch.zeros(max_num_utterances - num_utts, self.d_model, device=device)
                    audio_seq = torch.cat([audio_seq, padding], dim=0)
                    text_seq = torch.cat([text_seq, padding], dim=0)
                
                padded_audio.append(audio_seq)
                padded_text.append(text_seq)
                
                mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
                mask[num_utts:] = True
                attention_masks.append(mask)
            
            padded_audio = torch.stack(padded_audio)
            padded_text = torch.stack(padded_text)
            attention_masks = torch.stack(attention_masks)
            
            # Fusion
            if self.fusion_method == "concat":
                # Simple concatenation
                combined = torch.cat([padded_audio, padded_text], dim=-1)
                x = nn.Linear(self.d_model * 2, self.d_model, device=device)(combined)
            elif self.fusion_method == "cross_attention":
                x, _ = self.fusion_module(padded_audio, padded_text, attention_masks)
            elif self.fusion_method == "adaptive_gate":
                x, _ = self.fusion_module(padded_audio, padded_text)
            elif self.fusion_method == "hierarchical":
                x, _ = self.fusion_module(padded_audio, padded_text, attention_masks)
            else:
                x = padded_audio  # fallback
        
        else:
            # Audio-only (기존)
            combined_features = torch.cat([batch_wav2vec, q_type_embs, ttrs_expanded], dim=1)
            
            participant_sequences = []
            start_idx = 0
            
            for num_utts in num_utterances_list:
                end_idx = start_idx + num_utts
                participant_sequences.append(combined_features[start_idx:end_idx])
                start_idx = end_idx
            
            max_num_utterances = max(num_utterances_list)
            padded_sequences = []
            attention_masks = []
            
            for seq, num_utts in zip(participant_sequences, num_utterances_list):
                if num_utts < max_num_utterances:
                    padding = torch.zeros(max_num_utterances - num_utts, seq.size(1), device=device)
                    seq = torch.cat([seq, padding], dim=0)
                
                padded_sequences.append(seq)
                
                mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
                mask[num_utts:] = True
                attention_masks.append(mask)
            
            padded_sequences = torch.stack(padded_sequences)
            attention_masks = torch.stack(attention_masks)
            
            x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wav2vec = []
    batch_ttrs = []
    batch_q_type_ids = []
    batch_text_embed = []
    batch_linguistic = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        # ⭐ 텍스트 특징 추가
        if 'text_embedding' in utt and utt['text_embedding'] is not None:
            batch_text_embed.append(utt['text_embedding'])
        
        # ⭐ 언어학적 특징 추가
        linguistic_feat = [
            utt.get('word_count', 0),
            utt.get('avg_word_length', 0.0),
            utt.get('sentence_count', 0),
            utt.get('negation_count', 0),
            utt.get('first_person_count', 0)
        ]
        batch_linguistic.append(linguistic_feat)
    
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttrs = torch.FloatTensor(batch_ttrs)
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    batch_linguistic = torch.FloatTensor(np.array(batch_linguistic))
    
    result = {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list,
        'batch_linguistic': batch_linguistic
    }
    
    if batch_text_embed:
        result['batch_text_embed'] = torch.FloatTensor(np.array(batch_text_embed))
    
    return result


# =============================================================================
# 데이터 로드
# =============================================================================
def load_and_split_data():
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중... (텍스트 특징 포함)")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0, 
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 및 학습 (이전과 동일하지만 batch_linguistic 추가)
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            batch_linguistic = batch['batch_linguistic'].to(DEVICE)
            
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec, batch_ttrs, batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed, batch_linguistic
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(all_labels, all_probs, min_recall=MIN_RECALL_THRESHOLD)
        return {
            'loss': avg_loss, 'f1': f1, 'precision': precision, 'recall': recall,
            'labels': all_labels, 'probs': all_probs, 'preds': all_preds,
            'best_threshold': best_threshold, 'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss, 'f1': f1, 'precision': precision, 'recall': recall,
        'labels': all_labels, 'probs': all_probs, 'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.70):
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {'f1': f1, 'precision': precision, 'recall': recall, 'threshold': thresh}
    
    return best_threshold, best_metrics


def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_val_f1 = 0.0
    patience_counter = 0
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_specificity': []}
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작 - Full Multimodal (Audio + Text)")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            batch_linguistic = batch['batch_linguistic'].to(DEVICE)
            
            batch_text_embed = batch.get('batch_text_embed', None)
            if batch_text_embed is not None:
                batch_text_embed = batch_text_embed.to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec, batch_ttrs, batch_q_type_ids,
                batch['num_utterances_list'],
                batch_text_embed, batch_linguistic
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        val_results = evaluate(model, val_loader, criterion, threshold=0.5, return_all_thresholds=True)
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        scheduler.step()
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_results['loss']:.4f}")
        print(f"  Optimal Threshold: {best_thresh:.3f}")
        print(f"  Val F1: {optimal_f1:.4f}")
        print(f"  Val Prec: {optimal_prec:.4f}")
        print(f"  Val Recall: {optimal_rec:.4f}")
        print(f"  Val Spec: {specificity:.4f}")
        
        is_balanced = (optimal_prec > 0.20 and optimal_rec > 0.55 and specificity > 0.20)
        should_save = (optimal_f1 > best_val_f1) or (epoch == 0) or (is_balanced and optimal_f1 > 0.5)
        
        if should_save:
            best_val_f1 = optimal_f1
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_f1': best_val_f1,
                'optimal_threshold': best_thresh,
                'history': history,
                'fusion_method': FUSION_METHOD
            }, os.path.join(BASE_PATH, f'best_full_multimodal_model_{FUSION_METHOD}.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history, fusion_method):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1')
    axes[0, 1].set_title('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['val_precision'], label='Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(history['val_recall'], label='Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, f'training_history_full_multimodal_{fusion_method}.png'))
    plt.close()


def plot_confusion_matrix(labels, preds, fusion_method):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig(os.path.join(BASE_PATH, f'confusion_matrix_full_multimodal_{fusion_method}.png'))
    plt.close()


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    train_loader, val_loader, test_loader = load_and_split_data()
    
    model = MultimodalTransformerModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM,
        fusion_method=FUSION_METHOD,
        use_text_features=USE_TEXT_FEATURES
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 Full Multimodal Transformer (Audio + Text)")
    print(f"{'='*70}")
    print(f"파라미터: {total_params:,}")
    print(f"Fusion: {FUSION_METHOD}")
    print(f"Input Dim: {INPUT_DIM}")
    print(f"  - Wav2Vec: {WAV2VEC_DIM}")
    print(f"  - Text Embed: {TEXT_EMBED_DIM}")
    print(f"  - Q-Type: {Q_TYPE_EMBED_DIM}")
    print(f"  - TTR: {TTR_DIM}")
    print(f"  - Linguistic: {LINGUISTIC_FEATURES_DIM}")
    print(f"{'='*70}\n")
    
    history = train_model(model, train_loader, val_loader)
    plot_training_history(history, FUSION_METHOD)
    
    model_path = os.path.join(BASE_PATH, f'best_full_multimodal_model_{FUSION_METHOD}.pt')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        saved_threshold = checkpoint.get('optimal_threshold', 0.5)
        print(f"💡 최적 threshold: {saved_threshold:.3f}")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=saved_threshold)
        
        print(f"\n{'='*70}")
        print(f"📊 Test Set 평가")
        print(f"{'='*70}")
        print(f"Test F1: {test_results['f1']:.4f}")
        print(f"Test Precision: {test_results['precision']:.4f}")
        print(f"Test Recall: {test_results['recall']:.4f}")
        
        plot_confusion_matrix(test_results['labels'], test_results['preds'], FUSION_METHOD)
        
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(test_results['labels'], test_results['preds'],
                                    target_names=['Normal', 'Depression']))
        
        print(f"\n✅ 완료!")
    else:
        print(f"\n⚠️  모델 저장 안됨")

📂 데이터 로드 중... (텍스트 특징 포함)
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 Full Multimodal Transformer (Audio + Text)
파라미터: 3,404,193
Fusion: hierarchical
Input Dim: 1574
  - Wav2Vec: 768
  - Text Embed: 768
  - Q-Type: 32
  - TTR: 1
  - Linguistic: 5

📍 Using Focal Loss (alpha=0.5, gamma=1.5)

🚀 학습 시작 - Full Multimodal (Audio + Text)



Epoch 1/50: 100%|██████████| 14/14 [00:00<00:00, 16.96it/s, loss=0.127]



Epoch 1/50
  Train Loss: 0.1424
  Val Loss: 0.1150
  Optimal Threshold: 0.100
  Val F1: 0.5333
  Val Prec: 0.3636
  Val Recall: 1.0000
  Val Spec: 0.0000
  ✅ Best model saved! (F1: 0.5333)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 21.22it/s, loss=0.127]



Epoch 2/50
  Train Loss: 0.1238
  Val Loss: 0.1100
  Optimal Threshold: 0.100
  Val F1: 0.5333
  Val Prec: 0.3636
  Val Recall: 1.0000
  Val Spec: 0.0000
  ⏳ No improvement (1/20)
----------------------------------------------------------------------


...


Epoch 48/50
  Train Loss: 0.0848
  Val Loss: 0.1113
  Optimal Threshold: 0.600
  Val F1: 0.7333
  Val Prec: 0.6111
  Val Recall: 0.9167
  Val Spec: 0.6667
  ✅ Best model saved! (F1: 0.7333)
----------------------------------------------------------------------


Epoch 49/50: 100%|██████████| 14/14 [00:00<00:00, 23.22it/s, loss=0.026] 



Epoch 49/50
  Train Loss: 0.0865
  Val Loss: 0.1119
  Optimal Threshold: 0.460
  Val F1: 0.7097
  Val Prec: 0.5789
  Val Recall: 0.9167
  Val Spec: 0.6190
  ✅ Best model saved! (F1: 0.7097)
----------------------------------------------------------------------


Epoch 50/50: 100%|██████████| 14/14 [00:00<00:00, 22.64it/s, loss=0.11]  



Epoch 50/50
  Train Loss: 0.0864
  Val Loss: 0.1123
  Optimal Threshold: 0.460
  Val F1: 0.7097
  Val Prec: 0.5789
  Val Recall: 0.9167
  Val Spec: 0.6190
  ✅ Best model saved! (F1: 0.7097)
----------------------------------------------------------------------
💡 최적 threshold: 0.460

📊 Test Set 평가
Test F1: 0.5366
Test Precision: 0.4074
Test Recall: 0.7857

분류 보고서:
              precision    recall  f1-score   support

      Normal       0.84      0.50      0.63        32
  Depression       0.41      0.79      0.54        14

    accuracy                           0.59        46
   macro avg       0.62      0.64      0.58        46
weighted avg       0.71      0.59      0.60        46


✅ 완료!


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32

# ⭐ TTR 강화: 기존 1차원 → 6차원
TTR_ENHANCED_DIM = 6  # ttr, ttr_log, ttr_std, repetition_rate, unique_word_ratio, lexical_density

INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_ENHANCED_DIM  # 768 + 32 + 6 = 806

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# 불균형 처리
USE_FOCAL_LOSS = True
FOCAL_ALPHA = 0.45
FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.08

# Threshold
DYNAMIC_THRESHOLD = True
MIN_RECALL_THRESHOLD = 0.70

EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# ⭐ Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    """
    TTR 관련 강화된 특징 추출
    
    Returns:
        dict: {
            'ttr': 기본 TTR,
            'ttr_log': log-scaled TTR (긴 텍스트 보정),
            'repetition_rate': 반복 단어 비율,
            'unique_word_ratio': 고유 단어 비율,
            'lexical_density': 내용어 밀도,
            'word_length_variance': 단어 길이 분산
        }
    """
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    
    if len(tokens) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    # 1. 기본 TTR
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    
    # 2. Log-scaled TTR (긴 텍스트에서 TTR이 낮아지는 경향 보정)
    # TTR_log = types / log(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    # 3. Repetition rate (반복 단어 비율)
    # 2번 이상 등장하는 단어 비율
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    # 4. Unique word ratio (고유 단어 비율)
    # 1번만 등장하는 단어 비율 (높을수록 다양함)
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    # 5. Lexical density (내용어 밀도)
    # 기능어(function words)를 제외한 내용어 비율
    function_words = {
        'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
        'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'her', 'its', 'our', 'their',
        'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
        'this', 'that', 'these', 'those',
        'what', 'which', 'who', 'when', 'where', 'why', 'how'
    }
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    # 6. Word length variance (단어 길이 분산)
    # 우울증 환자는 단순한 단어 반복 → 분산 낮음
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr,
        'ttr_log': ttr_log,
        'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio,
        'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# ⭐ TTR-Enhanced Transformer Model
# =============================================================================
class TTREnhancedTransformerModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(TTREnhancedTransformerModel, self).__init__()
        
        self.d_model = d_model
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # ⭐ TTR Feature Projection (강화된 TTR 특징 처리)
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        # Input projection
        self.input_projection = nn.Linear(WAV2VEC_DIM + Q_TYPE_EMBED_DIM + 32, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttr_enhanced: [total_utterances, 6] ⭐ 강화된 TTR 특징
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size]
        """
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        
        # ⭐ TTR 특징 projection
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        # 결합: Wav2Vec + Q-type + TTR-projected
        combined_features = torch.cat([
            batch_wav2vec,
            q_type_embs,
            ttr_projected
        ], dim=1)
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Input Projection
        x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    """⭐ Enhanced TTR features 추출"""
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        # ⭐ Enhanced TTR features 추출
        text = utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'],
            ttr_features['ttr_log'],
            ttr_features['repetition_rate'],
            ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'],
            ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_and_split_data():
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중... (TTR Enhanced)")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0, 
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(
            all_labels, all_probs, min_recall=MIN_RECALL_THRESHOLD
        )
        return {
            'loss': avg_loss, 'f1': f1, 'precision': precision, 'recall': recall,
            'labels': all_labels, 'probs': all_probs, 'preds': all_preds,
            'best_threshold': best_threshold, 'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss, 'f1': f1, 'precision': precision, 'recall': recall,
        'labels': all_labels, 'probs': all_probs, 'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.70):
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if recall < min_recall:
            continue
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {'f1': f1, 'precision': precision, 'recall': recall, 'threshold': thresh}
    
    return best_threshold, best_metrics


# =============================================================================
# Training
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_val_f1 = 0.0
    patience_counter = 0
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_specificity': []}
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작 - TTR Enhanced Model")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        val_results = evaluate(model, val_loader, criterion, threshold=0.5, return_all_thresholds=True)
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        scheduler.step()
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_results['loss']:.4f}")
        print(f"  Optimal Threshold: {best_thresh:.3f}")
        print(f"  Val F1: {optimal_f1:.4f}")
        print(f"  Val Prec: {optimal_prec:.4f}")
        print(f"  Val Recall: {optimal_rec:.4f}")
        print(f"  Val Spec: {specificity:.4f}")
        
        is_balanced = (optimal_prec > 0.20 and optimal_rec > 0.55 and specificity > 0.20)
        should_save = (optimal_f1 > best_val_f1) or (epoch == 0) or (is_balanced and optimal_f1 > 0.5)
        
        if should_save:
            best_val_f1 = optimal_f1
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_f1': best_val_f1,
                'optimal_threshold': best_thresh,
                'history': history
            }, os.path.join(BASE_PATH, 'best_ttr_enhanced_model.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1')
    axes[0, 1].set_title('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['val_precision'], label='Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(history['val_recall'], label='Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, 'training_history_ttr_enhanced.png'))
    plt.close()


def plot_confusion_matrix(labels, preds):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig(os.path.join(BASE_PATH, 'confusion_matrix_ttr_enhanced.png'))
    plt.close()


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    train_loader, val_loader, test_loader = load_and_split_data()
    
    model = TTREnhancedTransformerModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 TTR Enhanced Transformer Model")
    print(f"{'='*70}")
    print(f"파라미터: {total_params:,}")
    print(f"Input Dim: {INPUT_DIM}")
    print(f"  - Wav2Vec: {WAV2VEC_DIM}")
    print(f"  - Q-Type: {Q_TYPE_EMBED_DIM}")
    print(f"  - TTR Enhanced: {TTR_ENHANCED_DIM}")
    print(f"\nTTR Features:")
    print(f"  1. Basic TTR")
    print(f"  2. Log-scaled TTR")
    print(f"  3. Repetition Rate")
    print(f"  4. Unique Word Ratio")
    print(f"  5. Lexical Density")
    print(f"  6. Word Length Variance")
    print(f"{'='*70}\n")
    
    history = train_model(model, train_loader, val_loader)
    plot_training_history(history)
    
    model_path = os.path.join(BASE_PATH, 'best_ttr_enhanced_model.pt')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        saved_threshold = checkpoint.get('optimal_threshold', 0.5)
        print(f"💡 최적 threshold: {saved_threshold:.3f}")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=saved_threshold)
        
        print(f"\n{'='*70}")
        print(f"📊 Test Set 평가")
        print(f"{'='*70}")
        print(f"Test F1: {test_results['f1']:.4f}")
        print(f"Test Precision: {test_results['precision']:.4f}")
        print(f"Test Recall: {test_results['recall']:.4f}")
        
        plot_confusion_matrix(test_results['labels'], test_results['preds'])
        
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(test_results['labels'], test_results['preds'],
                                    target_names=['Normal', 'Depression']))
        
        print(f"\n✅ 완료!")
    else:
        print(f"\n⚠️  모델 저장 안됨")

📂 데이터 로드 중... (TTR Enhanced)
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 TTR Enhanced Transformer Model
파라미터: 1,829,857
Input Dim: 806
  - Wav2Vec: 768
  - Q-Type: 32
  - TTR Enhanced: 6

TTR Features:
  1. Basic TTR
  2. Log-scaled TTR
  3. Repetition Rate
  4. Unique Word Ratio
  5. Lexical Density
  6. Word Length Variance

📍 Using Focal Loss (alpha=0.45, gamma=1.5)

🚀 학습 시작 - TTR Enhanced Model



Epoch 1/50: 100%|██████████| 14/14 [00:01<00:00, 10.80it/s, loss=0.125]



Epoch 1/50
  Train Loss: 0.1254
  Val Loss: 0.1176
  Optimal Threshold: 0.100
  Val F1: 0.5333
  Val Prec: 0.3636
  Val Recall: 1.0000
  Val Spec: 0.0000
  ✅ Best model saved! (F1: 0.5333)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:01<00:00, 13.34it/s, loss=0.125] 



Epoch 2/50
  Train Loss: 0.1202
  Val Loss: 0.1099
  Optimal Threshold: 0.100
  Val F1: 0.5333
  Val Prec: 0.3636
  Val Recall: 1.0000
  Val Spec: 0.0000
  ⏳ No improvement (1/15)
----------------------------------------------------------------------


...


Epoch 48/50
  Train Loss: 0.1048
  Val Loss: 0.1053
  Optimal Threshold: 0.360
  Val F1: 0.6486
  Val Prec: 0.4800
  Val Recall: 1.0000
  Val Spec: 0.3810
  ✅ Best model saved! (F1: 0.6486)
----------------------------------------------------------------------


Epoch 49/50: 100%|██████████| 14/14 [00:01<00:00, 13.64it/s, loss=0.177] 



Epoch 49/50
  Train Loss: 0.1069
  Val Loss: 0.1053
  Optimal Threshold: 0.360
  Val F1: 0.6486
  Val Prec: 0.4800
  Val Recall: 1.0000
  Val Spec: 0.3810
  ✅ Best model saved! (F1: 0.6486)
----------------------------------------------------------------------


Epoch 50/50: 100%|██████████| 14/14 [00:01<00:00, 12.69it/s, loss=0.143] 



Epoch 50/50
  Train Loss: 0.1081
  Val Loss: 0.1052
  Optimal Threshold: 0.360
  Val F1: 0.6486
  Val Prec: 0.4800
  Val Recall: 1.0000
  Val Spec: 0.3810
  ✅ Best model saved! (F1: 0.6486)
----------------------------------------------------------------------
💡 최적 threshold: 0.360

📊 Test Set 평가
Test F1: 0.4545
Test Precision: 0.3333
Test Recall: 0.7143

분류 보고서:
              precision    recall  f1-score   support

      Normal       0.75      0.38      0.50        32
  Depression       0.33      0.71      0.45        14

    accuracy                           0.48        46
   macro avg       0.54      0.54      0.48        46
weighted avg       0.62      0.48      0.49        46


✅ 완료!


In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.trial import TrialState
import json

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 기본 설정
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정 (Optuna로 조정될 항목 제외)
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optuna 설정
N_TRIALS = 30  # 50 -> 30으로 줄임
STUDY_NAME = "ttr_enhanced_balanced_f1"
OPTUNA_EPOCHS = 20  # Optuna trial당 epoch 수 (전체 학습보다 짧게)


# =============================================================================
# Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    """TTR 관련 강화된 특징 추출"""
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    
    if len(tokens) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
        'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
        'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'her', 'its', 'our', 'their',
        'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
        'this', 'that', 'these', 'those',
        'what', 'which', 'who', 'when', 'where', 'why', 'how'
    }
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr,
        'ttr_log': ttr_log,
        'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio,
        'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# TTR-Enhanced Transformer Model
# =============================================================================
class TTREnhancedTransformerModel(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(TTREnhancedTransformerModel, self).__init__()
        
        self.d_model = d_model
        
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        input_dim = WAV2VEC_DIM + q_type_embed_dim + 32
        self.input_projection = nn.Linear(input_dim, d_model)
        
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        combined_features = torch.cat([
            batch_wav2vec,
            q_type_embs,
            ttr_projected
        ], dim=1)
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        x = self.input_projection(padded_sequences)
        
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        self.data = []
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_q_type_ids.append(utt['q_type_id'])
        
        text = utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'],
            ttr_features['ttr_log'],
            ttr_features['repetition_rate'],
            ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'],
            ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_and_split_data():
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중... (TTR Enhanced + Optuna)")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                             collate_fn=collate_fn, num_workers=0, 
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 - Balanced F1 계산
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5):
    """
    평가 함수 - Balanced F1 (양성/음성 클래스의 F1 조화평균) 계산
    """
    model.eval()
    
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    
    # Threshold 적용
    all_preds = (np.array(all_probs) > threshold).astype(int)
    
    # 전체 F1
    overall_f1 = f1_score(all_labels, all_preds, average='binary')
    
    # 클래스별 F1
    f1_per_class = f1_score(all_labels, all_preds, average=None)
    f1_normal = f1_per_class[0]  # 음성 클래스
    f1_depression = f1_per_class[1]  # 양성 클래스
    
    # Balanced F1 (조화평균)
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # Confusion matrix for specificity
    tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, all_preds))
    fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, all_preds))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'loss': avg_loss,
        'overall_f1': overall_f1,
        'balanced_f1': balanced_f1,
        'f1_normal': f1_normal,
        'f1_depression': f1_depression,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, metric='balanced_f1'):
    """
    최적 threshold 찾기
    metric: 'balanced_f1', 'overall_f1', 'recall' 등
    """
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = {
        'threshold': 0.5,
        'balanced_f1': 0.0,
        'overall_f1': 0.0,
        'f1_normal': 0.0,
        'f1_depression': 0.0,
        'precision': 0.0,
        'recall': 0.0
    }
    
    for thresh in np.arange(0.1, 0.9, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        
        # 예측이 모두 같은 클래스면 skip
        if len(np.unique(preds)) < 2:
            continue
        
        overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
        
        # f1_per_class가 2개 미만이면 skip
        if len(f1_per_class) < 2:
            continue
            
        f1_normal = f1_per_class[0]
        f1_depression = f1_per_class[1]
        
        if f1_normal > 0 and f1_depression > 0:
            balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
        else:
            balanced_f1 = 0.0
        
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        # 메트릭 선택
        if metric == 'balanced_f1':
            score = balanced_f1
        elif metric == 'overall_f1':
            score = overall_f1
        elif metric == 'recall':
            score = recall
        else:
            score = balanced_f1
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh,
                'balanced_f1': balanced_f1,
                'overall_f1': overall_f1,
                'f1_normal': f1_normal,
                'f1_depression': f1_depression,
                'precision': precision,
                'recall': recall
            }
    
    # 아무것도 찾지 못한 경우, 기본 threshold로 계산
    if best_score == 0.0:
        preds = (np.array(probs) > 0.5).astype(int)
        
        if len(np.unique(preds)) >= 2:
            overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
            f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
            
            if len(f1_per_class) >= 2:
                f1_normal = f1_per_class[0]
                f1_depression = f1_per_class[1]
                
                if f1_normal > 0 and f1_depression > 0:
                    balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
                else:
                    balanced_f1 = 0.0
                
                best_metrics = {
                    'threshold': 0.5,
                    'balanced_f1': balanced_f1,
                    'overall_f1': overall_f1,
                    'f1_normal': f1_normal,
                    'f1_depression': f1_depression,
                    'precision': precision_score(labels, preds, zero_division=0),
                    'recall': recall_score(labels, preds, zero_division=0)
                }
    
    return best_threshold, best_metrics


# =============================================================================
# Optuna Objective
# =============================================================================
def objective(trial, train_loader, val_loader):
    """
    Optuna objective function - Balanced F1 최적화
    """
    # 하이퍼파라미터 샘플링
    d_model = trial.suggest_categorical('d_model', [128, 256, 384])
    nhead = trial.suggest_categorical('nhead', [4, 8])
    num_encoder_layers = trial.suggest_int('num_encoder_layers', 2, 4)
    dim_feedforward = trial.suggest_categorical('dim_feedforward', [256, 512, 1024])
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    
    focal_alpha = trial.suggest_float('focal_alpha', 0.3, 0.7)
    focal_gamma = trial.suggest_float('focal_gamma', 1.0, 3.0)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.15)
    
    # 모델 생성
    model = TTREnhancedTransformerModel(
        d_model=d_model,
        nhead=nhead,
        num_encoder_layers=num_encoder_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    ).to(DEVICE)
    
    # Loss & Optimizer
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Scheduler
    warmup_epochs = 3
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=OPTUNA_EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_balanced_f1 = 0.0
    patience_counter = 0
    max_patience = 8  # Optuna에서는 더 짧은 patience
    
    for epoch in range(OPTUNA_EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
        
        scheduler.step()
        
        # Validation
        val_results = evaluate(model, val_loader, criterion, threshold=0.5)
        
        # Threshold 최적화 (balanced_f1 기준)
        best_threshold, threshold_metrics = find_optimal_threshold(
            val_results['labels'], 
            val_results['probs'], 
            metric='balanced_f1'
        )
        
        balanced_f1 = threshold_metrics['balanced_f1']
        
        # Pruning (중간 평가) - 5 epoch 이후부터
        if epoch >= 5:
            trial.report(balanced_f1, epoch)
            
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        # Best model tracking
        if balanced_f1 > best_balanced_f1:
            best_balanced_f1 = balanced_f1
            patience_counter = 0
        else:
            patience_counter += 1
            
            if patience_counter >= max_patience:
                break
    
    return best_balanced_f1


# =============================================================================
# Training with Best Params
# =============================================================================
def train_with_best_params(best_params, train_loader, val_loader):
    """최적 하이퍼파라미터로 전체 학습"""
    print(f"\n{'='*70}")
    print(f"🚀 최적 하이퍼파라미터로 전체 학습 시작")
    print(f"{'='*70}")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    print(f"{'='*70}\n")
    
    # 모델 생성
    model = TTREnhancedTransformerModel(
        d_model=best_params['d_model'],
        nhead=best_params['nhead'],
        num_encoder_layers=best_params['num_encoder_layers'],
        dim_feedforward=best_params['dim_feedforward'],
        dropout=best_params['dropout']
    ).to(DEVICE)
    
    criterion = FocalLoss(
        alpha=best_params['focal_alpha'],
        gamma=best_params['focal_gamma'],
        label_smoothing=best_params['label_smoothing']
    )
    
    optimizer = optim.AdamW(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_balanced_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_balanced_f1': [],
        'val_overall_f1': [],
        'val_f1_normal': [],
        'val_f1_depression': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []
    }
    
    for epoch in range(NUM_EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate(model, val_loader, criterion, threshold=0.5)
        
        # Threshold 최적화
        opt_threshold, threshold_metrics = find_optimal_threshold(
            val_results['labels'],
            val_results['probs'],
            metric='balanced_f1'
        )
        
        # History 저장
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_results['loss'])
        history['val_balanced_f1'].append(threshold_metrics['balanced_f1'])
        history['val_overall_f1'].append(threshold_metrics['overall_f1'])
        history['val_f1_normal'].append(threshold_metrics['f1_normal'])
        history['val_f1_depression'].append(threshold_metrics['f1_depression'])
        history['val_precision'].append(threshold_metrics['precision'])
        history['val_recall'].append(threshold_metrics['recall'])
        
        # Specificity 계산
        opt_preds = (np.array(val_results['probs']) > opt_threshold).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], opt_preds))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], opt_preds))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_results['loss']:.4f}")
        print(f"  Optimal Threshold: {opt_threshold:.3f}")
        print(f"  Balanced F1: {threshold_metrics['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {threshold_metrics['overall_f1']:.4f}")
        print(f"  F1 Normal (0): {threshold_metrics['f1_normal']:.4f}")
        print(f"  F1 Depression (1): {threshold_metrics['f1_depression']:.4f}")
        print(f"  Precision: {threshold_metrics['precision']:.4f}")
        print(f"  Recall: {threshold_metrics['recall']:.4f}")
        print(f"  Specificity: {specificity:.4f}")
        
        # Best model 저장
        if threshold_metrics['balanced_f1'] > best_balanced_f1:
            best_balanced_f1 = threshold_metrics['balanced_f1']
            best_threshold = opt_threshold
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_params': best_params,
                'balanced_f1': best_balanced_f1,
                'optimal_threshold': best_threshold,
                'history': history,
                'threshold_metrics': threshold_metrics
            }, os.path.join(BASE_PATH, 'best_ttr_enhanced_optuna_model.pt'))
            
            print(f"  ✅ Best model saved! (Balanced F1: {best_balanced_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    return model, history, best_threshold


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Balanced F1
    axes[0, 1].plot(history['val_balanced_f1'], label='Balanced F1', color='purple', linewidth=2)
    axes[0, 1].plot(history['val_overall_f1'], label='Overall F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('F1 Scores (Balanced vs Overall)')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Class-wise F1
    axes[0, 2].plot(history['val_f1_normal'], label='F1 Normal (0)', color='blue')
    axes[0, 2].plot(history['val_f1_depression'], label='F1 Depression (1)', color='red')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('F1 Score')
    axes[0, 2].set_title('Class-wise F1 Scores')
    axes[0, 2].legend()
    axes[0, 2].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    # Specificity
    axes[1, 2].plot(history['val_specificity'], label='Specificity', color='orange')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Specificity')
    axes[1, 2].set_title('Specificity')
    axes[1, 2].legend()
    axes[1, 2].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, 'training_history_optuna.png'), dpi=300)
    plt.close()
    print(f"✅ Training history plot saved!")


def plot_confusion_matrix(labels, preds, title="Confusion Matrix"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Depression'],
                yticklabels=['Normal', 'Depression'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, 'confusion_matrix_optuna.png'), dpi=300)
    plt.close()
    print(f"✅ Confusion matrix saved!")


def plot_optuna_optimization(study):
    """Optuna 최적화 결과 시각화"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Optimization history
    trials = study.trials
    epochs = [trial.number for trial in trials if trial.state == TrialState.COMPLETE]
    values = [trial.value for trial in trials if trial.state == TrialState.COMPLETE]
    
    axes[0].plot(epochs, values, marker='o')
    axes[0].set_xlabel('Trial')
    axes[0].set_ylabel('Balanced F1')
    axes[0].set_title('Optimization History')
    axes[0].grid(True)
    
    # Best value over time
    best_values = []
    current_best = 0
    for val in values:
        current_best = max(current_best, val)
        best_values.append(current_best)
    
    axes[1].plot(epochs, best_values, marker='o', color='green')
    axes[1].set_xlabel('Trial')
    axes[1].set_ylabel('Best Balanced F1')
    axes[1].set_title('Best Value Over Time')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, 'optuna_optimization.png'), dpi=300)
    plt.close()
    print(f"✅ Optuna optimization plot saved!")


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 TTR Enhanced Transformer + Optuna (Balanced F1 Optimization)")
    print(f"{'='*70}\n")
    
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # Optuna Study 생성
    study = optuna.create_study(
        study_name=STUDY_NAME,
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    
    print(f"\n{'='*70}")
    print(f"🔍 Optuna 하이퍼파라미터 탐색 시작 (Trials: {N_TRIALS})")
    print(f"{'='*70}\n")
    
    # 최적화 실행
    study.optimize(
        lambda trial: objective(trial, train_loader, val_loader),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )
    
    # 최적 결과 출력
    print(f"\n{'='*70}")
    print(f"✅ Optuna 최적화 완료!")
    print(f"{'='*70}")
    print(f"Best Balanced F1: {study.best_value:.4f}")
    print(f"\nBest Hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    # 최적 파라미터 저장
    with open(os.path.join(BASE_PATH, 'best_params_optuna.json'), 'w') as f:
        json.dump(study.best_params, f, indent=2)
    print(f"\n✅ Best parameters saved to best_params_optuna.json")
    
    # Optuna 결과 시각화
    plot_optuna_optimization(study)
    
    # 최적 파라미터로 전체 학습
    model, history, best_threshold = train_with_best_params(
        study.best_params,
        train_loader,
        val_loader
    )
    
    # History 시각화
    plot_training_history(history)
    
    # Test Set 평가
    print(f"\n{'='*70}")
    print(f"📊 Test Set 평가 (최적 모델)")
    print(f"{'='*70}")
    
    model_path = os.path.join(BASE_PATH, 'best_ttr_enhanced_optuna_model.pt')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        best_threshold = checkpoint['optimal_threshold']
        
        print(f"💡 최적 threshold: {best_threshold:.3f}")
        
        test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"\n🎯 Test Set Results:")
        print(f"  Balanced F1: {test_results['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {test_results['overall_f1']:.4f}")
        print(f"  F1 Normal (0): {test_results['f1_normal']:.4f}")
        print(f"  F1 Depression (1): {test_results['f1_depression']:.4f}")
        print(f"  Precision: {test_results['precision']:.4f}")
        print(f"  Recall: {test_results['recall']:.4f}")
        print(f"  Specificity: {test_results['specificity']:.4f}")
        
        # Confusion Matrix
        plot_confusion_matrix(test_results['labels'], test_results['preds'], title="Test Set Confusion Matrix")
        
        # Classification Report
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(
            test_results['labels'],
            test_results['preds'],
            target_names=['Normal', 'Depression'],
            digits=4
        ))
        
        print(f"\n{'='*70}")
        print(f"✅ 모든 과정 완료!")
        print(f"{'='*70}\n")
    else:
        print(f"\n⚠️  모델 파일을 찾을 수 없습니다.")

[I 2025-12-08 22:53:17,992] A new study created in memory with name: ttr_enhanced_balanced_f1



🤖 TTR Enhanced Transformer + Optuna (Balanced F1 Optimization)

📂 데이터 로드 중... (TTR Enhanced + Optuna)
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🔍 Optuna 하이퍼파라미터 탐색 시작 (Trials: 30)



  0%|          | 0/30 [00:00<?, ?it/s]

[I 2025-12-08 22:53:32,717] Trial 0 finished with value: 0.631578947368421 and parameters: {'d_model': 384, 'nhead': 4, 'num_encoder_layers': 2, 'dim_feedforward': 1024, 'dropout': 0.14347652021242882, 'learning_rate': 4.29960412574832e-05, 'weight_decay': 0.0007513264074056214, 'focal_alpha': 0.5401723040856345, 'focal_gamma': 1.072235076022304, 'label_smoothing': 0.08473177310809515}. Best is trial 0 with value: 0.631578947368421.
[I 2025-12-08 22:53:48,036] Trial 1 finished with value: 0.7142857142857142 and parameters: {'d_model': 384, 'nhead': 8, 'num_encoder_layers': 4, 'dim_feedforward': 256, 'dropout': 0.24054158230123326, 'learning_rate': 3.8025721355801666e-05, 'weight_decay': 0.0007103899282781471, 'focal_alpha': 0.3407439325708209, 'focal_gamma': 2.7651944904398063, 'label_smoothing': 0.14627893172869322}. Best is trial 1 with value: 0.7142857142857142.
[I 2025-12-08 22:54:07,889] Trial 2 finished with value: 0.676056338028169 and parameters: {'d_model': 256, 'nhead': 4, 'n

Epoch 1/50: 100%|██████████| 14/14 [00:01<00:00, 11.40it/s, loss=0.0332]


...


Epoch 33/50
  Train Loss: 0.0340
  Val Loss: 0.0532
  Optimal Threshold: 0.340
  Balanced F1: 0.7216 ⭐
  Overall F1: 0.6897
  F1 Normal (0): 0.7568
  F1 Depression (1): 0.6897
  Precision: 0.5882
  Recall: 0.8333
  Specificity: 0.6667
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 34/50: 100%|██████████| 14/14 [00:01<00:00, 12.53it/s, loss=0.034] 



Epoch 34/50
  Train Loss: 0.0367
  Val Loss: 0.0548
  Optimal Threshold: 0.320
  Balanced F1: 0.7259 ⭐
  Overall F1: 0.7097
  F1 Normal (0): 0.7429
  F1 Depression (1): 0.7097
  Precision: 0.5789
  Recall: 0.9167
  Specificity: 0.6190
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 35/50: 100%|██████████| 14/14 [00:01<00:00, 12.63it/s, loss=0.015] 



Epoch 35/50
  Train Loss: 0.0331
  Val Loss: 0.0551
  Optimal Threshold: 0.340
  Balanced F1: 0.7216 ⭐
  Overall F1: 0.6897
  F1 Normal (0): 0.7568
  F1 Depression (1): 0.6897
  Precision: 0.5882
  Recall: 0.8333
  Specificity: 0.6667
  ⏳ No improvement (15/15)

⚠️  Early stopping!
✅ Training history plot saved!

📊 Test Set 평가 (최적 모델)
💡 최적 threshold: 0.420

🎯 Test Set Results:
  Balanced F1: 0.6623 ⭐
  Overall F1: 0.6154
  F1 Normal (0): 0.7170
  F1 Depression (1): 0.6154
  Precision: 0.4800
  Recall: 0.8571
  Specificity: 0.5938
✅ Confusion matrix saved!

분류 보고서:
              precision    recall  f1-score   support

      Normal     0.9048    0.5938    0.7170        32
  Depression     0.4800    0.8571    0.6154        14

    accuracy                         0.6739        46
   macro avg     0.6924    0.7254    0.6662        46
weighted avg     0.7755    0.6739    0.6861        46


✅ 모든 과정 완료!

